# 03 통합 분석 v5

이 노트북은 `old v4` 폴더의 최신 EDA v2, 해석형 특징 v1, ML 학습·평가 v4 코드를 순서대로 통합한 버전입니다.

- 입력: `구축 데이터셋_v4`
- 출력: `머신러닝_분석결과_v5`
- 실행 순서: PART A → PART B → PART C
- 원본 v4 계산 로직은 유지하고 경로·버전명만 v5로 갱신했습니다.


---

# PART A — 탐색적 데이터 분석(EDA)

원본: `03_2_eda_analysis\03_2_eda_analysis_v2.ipynb`


# PART A. v5 보이스피싱 데이터 EDA

`구축 데이터셋_v4`의 데이터 품질, 정상상담과 보이스피싱의 차이, 사기유형·사칭·요구행동·심리전략·금액·대화구간을 탐색합니다.

이 노트북은 모델을 학습하지 않습니다. ML 전에 데이터의 구조와 편향을 확인하고, 보고서·대시보드에 사용할 표와 그림을 만드는 단계입니다.

In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow seaborn matplotlib koreanize-matplotlib scikit-learn openpyxl

In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from sklearn.feature_extraction.text import CountVectorizer
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 설정값

In [ ]:
# 2. 입력·출력 경로
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v4'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
DASHBOARD_ROOT = DATASET_ROOT / '03_dashboard_tables'
OUTPUT_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v5' / '01_EDA'
TABLE_ROOT = OUTPUT_ROOT / '01_분석표'
FIGURE_ROOT = OUTPUT_ROOT / '02_그래프'
REVIEW_ROOT = OUTPUT_ROOT / '03_검토대상'
REPORT_ROOT = OUTPUT_ROOT / '04_보고서'
for folder in [TABLE_ROOT, FIGURE_ROOT, REVIEW_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
TOP_N = 15
SEED = 42
assert STANDARD_ROOT.exists() and ML_ROOT.exists(), f'구축 데이터셋_v4 경로를 확인하세요: {DATASET_ROOT}'
print('입력:', DATASET_ROOT)
print('출력:', OUTPUT_ROOT)

## 2. 데이터 불러오기

In [ ]:
# 3. CSV보다 빠른 Parquet을 우선 사용합니다.
def read_table(folder, name, required=True):
    parquet_path = folder / f'{name}.parquet'
    csv_path = folder / f'{name}.csv'
    if parquet_path.exists(): return pd.read_parquet(parquet_path)
    if csv_path.exists(): return pd.read_csv(csv_path, encoding='utf-8-sig')
    if required: raise FileNotFoundError(f'{name}을 찾지 못했습니다: {folder}')
    return pd.DataFrame()
tables = {
 'vp_files':read_table(STANDARD_ROOT,'vp_files'),
 'vp_cases':read_table(STANDARD_ROOT,'vp_cases'),
 'vp_utterances':read_table(STANDARD_ROOT,'vp_utterances'),
 'vp_impersonations':read_table(STANDARD_ROOT,'vp_impersonations'),
 'vp_requested_actions':read_table(STANDARD_ROOT,'vp_requested_actions'),
 'vp_strategy_events':read_table(STANDARD_ROOT,'vp_strategy_events'),
 'vp_amount_events':read_table(STANDARD_ROOT,'vp_amount_events'),
 'normal_finance_calls':read_table(STANDARD_ROOT,'normal_finance_calls'),
 'fraud_detection_ml':read_table(ML_ROOT,'fraud_detection_ml'),
 'fraud_type_ml':read_table(ML_ROOT,'fraud_type_ml'),
 'segment_detection_ml':read_table(ML_ROOT,'segment_detection_ml'),
 'case_clustering_ml':read_table(ML_ROOT,'case_clustering_ml'),
 'dashboard_case_summary':read_table(DASHBOARD_ROOT,'dashboard_case_summary',required=False),
}
inventory=pd.DataFrame([{'테이블':name,'행수':len(df),'컬럼수':len(df.columns)} for name,df in tables.items()])
display(inventory); inventory.to_csv(TABLE_ROOT/'00_테이블_목록.csv',index=False,encoding='utf-8-sig')

## 3. 데이터 품질·결측·중복 확인

In [ ]:
# 4. 테이블별 품질 요약
pk_map={'vp_files':'file_id','vp_cases':'case_id','vp_utterances':'turn_id',
        'vp_impersonations':'impersonation_id','vp_requested_actions':'action_id',
        'vp_strategy_events':'strategy_event_id','vp_amount_events':'amount_event_id'}
quality_rows=[]; missing_rows=[]
for name,df in tables.items():
    if df.empty: continue
    pk=pk_map.get(name); pk_dup=int(df[pk].duplicated().sum()) if pk in df else np.nan
    quality_rows.append({'테이블':name,'행수':len(df),'완전중복행':int(df.duplicated().sum()),
                         '기본키':pk or '없음','기본키중복':pk_dup,'전체결측셀':int(df.isna().sum().sum())})
    for col,ratio in df.isna().mean().sort_values(ascending=False).items():
        if ratio>0: missing_rows.append({'테이블':name,'컬럼':col,'결측수':int(df[col].isna().sum()),'결측률':ratio})
quality_df=pd.DataFrame(quality_rows); missing_df=pd.DataFrame(missing_rows)
display(quality_df); display(missing_df.head(30))
quality_df.to_csv(TABLE_ROOT/'01_데이터품질_요약.csv',index=False,encoding='utf-8-sig')
missing_df.to_csv(TABLE_ROOT/'02_컬럼별_결측률.csv',index=False,encoding='utf-8-sig')
assert quality_df['기본키중복'].fillna(0).eq(0).all(), '기본키 중복이 발견되었습니다.'

In [ ]:
# 5. 빈 전사·역할·품질 플래그 확인
cases=tables['vp_cases'].copy(); utter=tables['vp_utterances'].copy()
text_col=next((c for c in ['raw_full_text','normalized_full_text','full_text'] if c in cases),None)
empty_cases=int(cases[text_col].fillna('').str.strip().eq('').sum()) if text_col else np.nan
quality_tables=[]
if 'quality_flag' in cases:
    quality_tables.append(cases['quality_flag'].fillna('MISSING').value_counts().rename_axis('품질상태').reset_index(name='사건수'))
if 'role' in utter:
    quality_tables.append(utter['role'].fillna('MISSING').value_counts().rename_axis('화자역할').reset_index(name='발화수'))
print('빈 사건 전사:',empty_cases)
for i,frame in enumerate(quality_tables): display(frame); frame.to_csv(TABLE_ROOT/f'03_품질분포_{i+1}.csv',index=False,encoding='utf-8-sig')

## 4. 정상상담과 보이스피싱의 분포·길이·출처 편향

In [ ]:
# 6. 정상상담과 보이스피싱 기본 비교
det=tables['fraud_detection_ml'].copy()
det['text_length']=det['model_input_text'].fillna('').str.len()
det['word_count']=det['model_input_text'].fillna('').str.split().str.len()
class_summary=det['fraud_label'].value_counts().rename_axis('구분').reset_index(name='건수')
class_summary['비율']=class_summary['건수']/class_summary['건수'].sum()
length_summary=det.groupby('fraud_label')[['text_length','word_count']].agg(['count','mean','median','std','min','max']).round(2)
display(class_summary); display(length_summary)
class_summary.to_csv(TABLE_ROOT/'04_정상사기_클래스분포.csv',index=False,encoding='utf-8-sig')
length_summary.to_csv(TABLE_ROOT/'05_정상사기_텍스트길이.csv',encoding='utf-8-sig')
fig,axes=plt.subplots(1,3,figsize=(18,5))
sns.countplot(data=det,x='fraud_label',ax=axes[0]); axes[0].set_title('정상상담과 보이스피싱 건수')
sns.boxplot(data=det,x='fraud_label',y='text_length',showfliers=False,ax=axes[1]); axes[1].set_title('텍스트 길이')
sns.histplot(data=det,x='text_length',hue='fraud_label',element='step',stat='density',common_norm=False,ax=axes[2]); axes[2].set_title('텍스트 길이 밀도')
for ax in axes: ax.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'01_정상사기_분포와길이.png',dpi=170,bbox_inches='tight'); plt.show()

In [ ]:
# 7. 출처·정상상담 분야·공식 split 확인
for col,title in [('source_group','출처'),('financial_topic','금융상담주제'),('original_split','원본분리')]:
    if col not in det: continue
    frame=pd.crosstab(det[col].fillna('MISSING'),det['fraud_label'],margins=True)
    display(frame); frame.to_csv(TABLE_ROOT/f'06_{title}_교차표.csv',encoding='utf-8-sig')
if 'financial_topic' in det:
    topic=det.groupby(['fraud_label','financial_topic']).size().reset_index(name='건수')
    plt.figure(figsize=(11,6)); sns.barplot(data=topic,y='financial_topic',x='건수',hue='fraud_label')
    plt.title('정상상담 분야와 보이스피싱 원본분류'); plt.tight_layout()
    plt.savefig(FIGURE_ROOT/'02_출처와_상담주제.png',dpi=170,bbox_inches='tight'); plt.show()

In [ ]:
# 8. 클래스별 주요 단어·2-gram: 출처 문체 차이도 함께 포함될 수 있습니다.
texts=det['model_input_text'].fillna('').astype(str)
vectorizer=CountVectorizer(binary=True,ngram_range=(1,2),min_df=5,max_features=20000,token_pattern=r'(?u)\b[^\s]{2,}\b')
matrix=vectorizer.fit_transform(texts); terms=np.array(vectorizer.get_feature_names_out())
labels=det['fraud_label'].to_numpy(); classes=sorted(pd.unique(labels))
word_rows=[]
if len(classes)==2:
    rates={}
    for label in classes: rates[label]=np.asarray(matrix[labels==label].mean(axis=0)).ravel()
    diff=rates['VOICE_PHISHING']-rates['LEGITIMATE_FINANCIAL_CALL']
    for idx in diff.argsort()[-30:][::-1]: word_rows.append({'구분':'보이스피싱 상대고빈도','단어':terms[idx],'출현율차이':diff[idx]})
    for idx in diff.argsort()[:30]: word_rows.append({'구분':'정상상담 상대고빈도','단어':terms[idx],'출현율차이':diff[idx]})
word_df=pd.DataFrame(word_rows); display(word_df.head(30))
word_df.to_csv(TABLE_ROOT/'07_정상사기_상대고빈도단어.csv',index=False,encoding='utf-8-sig')

## 5. 보이스피싱 유형·사칭·요구행동

In [ ]:
# 9. 사건유형·사칭·요구행동 단순 분포
type_df=tables['fraud_type_ml']; imp=tables['vp_impersonations']; actions=tables['vp_requested_actions']
type_map=type_df[['case_id','supervised_target']].drop_duplicates('case_id')
def save_count(df,col,name,top_n=None):
    if col not in df: return pd.DataFrame()
    result=df[col].fillna('MISSING').value_counts().rename_axis(col).reset_index(name='건수')
    result['비율']=result['건수']/result['건수'].sum()
    result.to_csv(TABLE_ROOT/f'{name}.csv',index=False,encoding='utf-8-sig')
    display(result.head(top_n or len(result))); return result
type_count=save_count(type_df,'supervised_target','08_보이스피싱_유형분포')
imp_group_col=next((c for c in ['impersonation_group','primary_impersonation_group'] if c in imp),None)
imp_sub_col=next((c for c in ['impersonation_subtype','claimed_org_name'] if c in imp),None)
action_col=next((c for c in ['action_type','primary_requested_action'] if c in actions),None)
if imp_group_col: save_count(imp,imp_group_col,'09_사칭대분류_분포',TOP_N)
if imp_sub_col: save_count(imp,imp_sub_col,'10_사칭세부유형_분포',TOP_N)
if action_col: save_count(actions,action_col,'11_요구행동_분포',TOP_N)

In [ ]:
# 의미 있는 단어·2-gram·의미 범주 분석

# 1. 분석에서 제외할 구어체·불용어
stopwords = {
    '거고요', '네네', '그러니까', '때문에', '대해서',
    '거예요', '겁니다', '그런', '저희가', '이런',
    '아니', '이렇게', '해서', '일단은', '쪽에서',
    '건데', '돼요', '여보세요', '본인이', '본인께서',
    '그리고', '그런데', '그러면', '그래서', '그냥',
    '제가', '지금', '일단'
}

# 2. 의미 있는 보이스피싱 상대 고빈도 단어
meaningful_word_df = word_df[
    (word_df['구분'] == '보이스피싱 상대고빈도')
    & (~word_df['단어'].isin(stopwords))
    & (word_df['단어'].str.len() >= 2)
].copy()

meaningful_word_df = meaningful_word_df.nlargest(
    20,
    '출현율차이'
)

display(meaningful_word_df)

meaningful_word_df.to_csv(
    TABLE_ROOT / '08_보이스피싱_의미단어.csv',
    index=False,
    encoding='utf-8-sig'
)

# 3. 의미 단어 막대그래프
plot_word_df = meaningful_word_df.sort_values('출현율차이')

plt.figure(figsize=(10, 7))

sns.barplot(
    data=plot_word_df,
    x='출현율차이',
    y='단어',
    color='#d95f5f'
)

plt.title('정상 금융상담보다 보이스피싱에서 자주 등장한 단어')
plt.xlabel('문서 출현율 차이')
plt.ylabel('단어')
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '11_보이스피싱_의미단어.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()


# 4. 의미 범주 분류
category_patterns = {
    '수사·범죄 서사': (
        r'사건|불법|범죄|피해자|조사|수사|'
        r'현장|검찰|검사|경찰|수사관|체포|구속'
    ),
    '신원·명의 확인': (
        r'명의|본인|신분|주민|개인정보|생년월일'
    ),
    '금융수단·계좌': (
        r'통장|계좌|은행|카드|잔액|송금|이체|입금'
    ),
    '행동통제·연락': (
        r'연락|전화|끊지|말하지|비밀|'
        r'하세요|하셔야|따라'
    ),
    '금전요구': (
        r'돈|금액|수수료|세금|보증금|'
        r'공탁금|납부|현금'
    ),
    '긴급성·위협': (
        r'긴급|즉시|빨리|당장|위험|'
        r'처벌|압류|고소|고발'
    ),
    '이익·혜택 제안': (
        r'대출|승인|환급|지원금|혜택|저금리'
    )
}


def classify_word_category(word):
    for category, pattern in category_patterns.items():
        if re.search(pattern, str(word)):
            return category
    return '기타'


meaningful_word_df['의미범주'] = (
    meaningful_word_df['단어']
    .map(classify_word_category)
)

category_summary_df = (
    meaningful_word_df
    .groupby('의미범주', as_index=False)
    .agg(
        단어수=('단어', 'count'),
        출현율차이_합계=('출현율차이', 'sum'),
        대표단어=(
            '단어',
            lambda values: ', '.join(values.head(5))
        )
    )
    .sort_values(
        '출현율차이_합계',
        ascending=False
    )
)

display(category_summary_df)

category_summary_df.to_csv(
    TABLE_ROOT / '09_보이스피싱_단어의미범주.csv',
    index=False,
    encoding='utf-8-sig'
)

# 5. 의미 범주 그래프
plot_category_df = category_summary_df.sort_values(
    '출현율차이_합계'
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=plot_category_df,
    x='출현율차이_합계',
    y='의미범주',
    color='#e78b3e'
)

plt.title('보이스피싱 상대 고빈도 단어의 의미 범주')
plt.xlabel('상위 단어 출현율 차이 합계')
plt.ylabel('의미 범주')
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '12_보이스피싱_단어의미범주.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()


# 6. 의미 있는 2-gram 다시 계산
bigram_vectorizer = CountVectorizer(
    binary=True,
    ngram_range=(2, 2),
    min_df=5,
    max_features=30000,
    token_pattern=r'(?u)\b[^\s]{2,}\b'
)

bigram_matrix = bigram_vectorizer.fit_transform(
    det['model_input_text'].fillna('').astype(str)
)

bigram_terms = np.array(
    bigram_vectorizer.get_feature_names_out()
)

labels = det['fraud_label'].to_numpy()

fraud_rate = np.asarray(
    bigram_matrix[labels == 'VOICE_PHISHING']
    .mean(axis=0)
).ravel()

normal_rate = np.asarray(
    bigram_matrix[
        labels == 'LEGITIMATE_FINANCIAL_CALL'
    ]
    .mean(axis=0)
).ravel()

bigram_difference = fraud_rate - normal_rate

bigram_df = pd.DataFrame({
    '2그램': bigram_terms,
    '보이스피싱_출현율': fraud_rate,
    '정상상담_출현율': normal_rate,
    '출현율차이': bigram_difference
})

# 불용어가 포함된 2-gram 제외
bigram_df['불용어포함'] = bigram_df['2그램'].map(
    lambda value: any(
        token in stopwords
        for token in str(value).split()
    )
)

meaningful_bigram_df = (
    bigram_df[
        (~bigram_df['불용어포함'])
        & (bigram_df['출현율차이'] > 0)
    ]
    .nlargest(20, '출현율차이')
    .drop(columns='불용어포함')
)

display(meaningful_bigram_df)

meaningful_bigram_df.to_csv(
    TABLE_ROOT / '10_보이스피싱_의미있는_2그램.csv',
    index=False,
    encoding='utf-8-sig'
)

# 7. 2-gram 그래프
plot_bigram_df = meaningful_bigram_df.sort_values(
    '출현율차이'
)

plt.figure(figsize=(11, 8))

sns.barplot(
    data=plot_bigram_df,
    x='출현율차이',
    y='2그램',
    color='#7b68ae'
)

plt.title('정상 금융상담보다 보이스피싱에서 자주 등장한 2-gram')
plt.xlabel('문서 출현율 차이')
plt.ylabel('2-gram')
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '13_보이스피싱_의미있는_2그램.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()

print('의미 단어·범주·2-gram 분석 완료')

In [ ]:
# 10. 사기유형 × 사칭기관, 사기유형 × 요구행동
def event_crosstab(event_df,event_col,filename):
    if event_col is None or event_df.empty: return pd.DataFrame()
    merged=event_df.merge(type_map,on='case_id',how='inner')
    count=pd.crosstab(merged['supervised_target'],merged[event_col])
    ratio=pd.crosstab(merged['supervised_target'],merged[event_col],normalize='index').round(4)
    count.to_csv(TABLE_ROOT/f'{filename}_건수.csv',encoding='utf-8-sig')
    ratio.to_csv(TABLE_ROOT/f'{filename}_행비율.csv',encoding='utf-8-sig')
    display(count); display(ratio); return count
imp_cross=event_crosstab(imp,imp_sub_col,'12_사기유형_사칭세부유형')
action_cross=event_crosstab(actions,action_col,'13_사기유형_요구행동')
if not action_cross.empty:
    top=action_cross.sum().nlargest(min(12,len(action_cross.columns))).index
    plt.figure(figsize=(13,4)); sns.heatmap(action_cross[top],annot=True,fmt='g',cmap='Blues')
    plt.title('사기유형 × 요구행동'); plt.tight_layout(); plt.savefig(FIGURE_ROOT/'03_사기유형_요구행동.png',dpi=170); plt.show()

In [ ]:
# 사기유형별 사칭 대상 구성비 시각화

# 동일 사건에서 동일 기관이 여러 번 추출된 경우 한 번만 계산
impersonation_case_df = (
    imp[['case_id', imp_sub_col]]
    .dropna()
    .drop_duplicates(['case_id', imp_sub_col])
    .merge(
        type_map,
        on='case_id',
        how='inner'
    )
)

# 사칭 대상 건수
impersonation_count_df = pd.crosstab(
    impersonation_case_df['supervised_target'],
    impersonation_case_df[imp_sub_col]
)

# 각 사기유형 안에서의 구성비(%)
impersonation_ratio_df = pd.crosstab(
    impersonation_case_df['supervised_target'],
    impersonation_case_df[imp_sub_col],
    normalize='index'
) * 100

# 한글 이름
impersonation_name_map = {
    'BANK': '은행',
    'BUSINESS_CONTACT': '거래처',
    'CAPITAL_COMPANY': '캐피탈',
    'CARD_COMPANY': '카드회사',
    'COURT': '법원',
    'DELIVERY_COMPANY': '택배회사',
    'FINANCIAL_ASSOCIATION': '금융협회',
    'FSS': '금융감독원',
    'KIDNAPPING_CLAIM': '납치 주장',
    'LOAN_COMPANY': '대출회사',
    'PERSONAL_CONTACT': '지인·가족',
    'POLICE': '경찰',
    'POST_OFFICE': '우체국',
    'PROSECUTION': '검찰',
    'RECRUITER': '채용담당자',
    'SAVINGS_BANK': '저축은행',
    'TAX_AUTHORITY': '세무기관',
    'TELECOM': '통신회사'
}

fraud_type_name_map = {
    'INSTITUTION_IMPERSONATION': '수사기관 사칭형',
    'LOAN_FRAUD': '대출사기형'
}

impersonation_ratio_ko_df = (
    impersonation_ratio_df
    .rename(
        index=fraud_type_name_map,
        columns=impersonation_name_map
    )
)

display(impersonation_ratio_ko_df.round(2))

# 결과 저장
impersonation_count_df.to_csv(
    TABLE_ROOT / '사기유형별_사칭대상_건수.csv',
    encoding='utf-8-sig'
)

impersonation_ratio_ko_df.to_csv(
    TABLE_ROOT / '사기유형별_사칭대상_구성비.csv',
    encoding='utf-8-sig'
)


# 1. 주요 사칭 대상 막대그래프
major_impersonations = [
    '은행',
    '카드회사',
    '검찰',
    '경찰',
    '캐피탈',
    '저축은행',
    '대출회사',
    '금융감독원'
]

available_impersonations = [
    column
    for column in major_impersonations
    if column in impersonation_ratio_ko_df.columns
]

bar_df = (
    impersonation_ratio_ko_df[available_impersonations]
    .reset_index()
    .rename(columns={'supervised_target': '사기유형'})
    .melt(
        id_vars='사기유형',
        var_name='사칭대상',
        value_name='구성비'
    )
)

plt.figure(figsize=(14, 7))

ax = sns.barplot(
    data=bar_df,
    x='사칭대상',
    y='구성비',
    hue='사기유형',
    palette=['#4c72b0', '#dd8452']
)

plt.title('보이스피싱 유형별 주요 사칭 대상')
plt.xlabel('사칭 대상')
plt.ylabel('사칭 대상 구성비(%)')
plt.xticks(rotation=20)
plt.legend(title='보이스피싱 유형')

# 막대 위에 비율 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=3,
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '사기유형별_주요사칭대상_막대그래프.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()


# 2. 전체 사칭 대상 히트맵
plt.figure(figsize=(18, 5))

sns.heatmap(
    impersonation_ratio_ko_df,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar_kws={'label': '구성비(%)'}
)

plt.title('보이스피싱 유형별 사칭 대상 구성비')
plt.xlabel('사칭 대상')
plt.ylabel('보이스피싱 유형')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '사기유형별_사칭대상_히트맵.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()

print('사칭 대상 그래프 저장 완료')

In [ ]:
# 수사기관형에서 BANK로 추출된 사례 확인

bank_cases = (
    imp[
        imp[imp_sub_col].eq('BANK')
    ]
    .merge(
        type_map,
        on='case_id',
        how='inner'
    )
)

bank_cases = bank_cases[
    bank_cases['supervised_target']
    .eq('INSTITUTION_IMPERSONATION')
]

check_columns = [
    column for column in [
        'case_id',
        'impersonation_group',
        'impersonation_subtype',
        'claimed_org_name',
        'claimed_role_title',
        'is_primary_impersonation',
        'evidence_text'
    ]
    if column in bank_cases.columns
]

display(
    bank_cases[check_columns]
    .sort_values(
        'is_primary_impersonation',
        ascending=False
    )
    .head(50)
)

In [ ]:
primary_imp = imp.copy()

if 'is_primary_impersonation' in primary_imp.columns:
    primary_imp = primary_imp[
        primary_imp['is_primary_impersonation'].eq(True)
    ]

primary_imp = primary_imp.merge(
    type_map,
    on='case_id',
    how='inner'
)

primary_ratio = pd.crosstab(
    primary_imp['supervised_target'],
    primary_imp[imp_sub_col],
    normalize='index'
) * 100

display(primary_ratio.round(1))

In [ ]:
# 사건 단위 사칭 대상 검토표 생성

imp_case = imp.copy()

# 숫자·문자형 정리
imp_case['mention_count'] = pd.to_numeric(
    imp_case.get('mention_count', 0),
    errors='coerce'
).fillna(0)

imp_case['first_mention_turn'] = pd.to_numeric(
    imp_case.get('first_mention_turn', 999999),
    errors='coerce'
).fillna(999999)

imp_case['extraction_confidence'] = pd.to_numeric(
    imp_case.get('extraction_confidence', 0),
    errors='coerce'
).fillna(0)

# 범인이 자신의 소속이라고 명시적으로 주장했는지
imp_case['명시적_자기사칭'] = (
    imp_case.get(
        'identity_claim_type',
        pd.Series('', index=imp_case.index)
    ).eq('CALLER_SELF_CLAIM')
    &
    imp_case.get(
        'evidence_role',
        pd.Series('', index=imp_case.index)
    ).eq('OFFENDER')
)

# 직책이 함께 등장했는지
role_title = imp_case.get(
    'claimed_role_title_normalized',
    pd.Series('', index=imp_case.index)
).fillna('').astype(str)

imp_case['직책존재'] = role_title.str.strip().ne('')

case_rows = []

for case_id, group in imp_case.groupby('case_id'):

    group = group.sort_values('first_mention_turn')

    # 사건에서 가장 먼저 등장한 사칭기관
    access_subtype = (
        group.iloc[0]['impersonation_subtype']
        if len(group)
        else 'UNKNOWN'
    )

    # 범인이 명시적으로 자기 소속이라고 주장한 후보
    explicit_group = group[
        group['명시적_자기사칭']
    ].copy()

    if len(explicit_group):

        # 반복 횟수, 직책, 신뢰도가 높은 후보를 우선
        explicit_group = explicit_group.sort_values(
            [
                'mention_count',
                '직책존재',
                'extraction_confidence',
                'first_mention_turn'
            ],
            ascending=[False, False, False, True]
        )

        primary_row = explicit_group.iloc[0]
        primary_subtype = primary_row[
            'impersonation_subtype'
        ]
        primary_evidence = primary_row.get(
            'evidence_text',
            ''
        )
        primary_confidence = primary_row[
            'extraction_confidence'
        ]

    else:
        # 명시적 자기소개가 없으면 억지로 확정하지 않음
        primary_subtype = 'UNKNOWN'
        primary_evidence = ''
        primary_confidence = 0.0

    all_subtypes = list(
        dict.fromkeys(
            group['impersonation_subtype']
            .dropna()
            .astype(str)
            .tolist()
        )
    )

    secondary_subtypes = [
        subtype
        for subtype in all_subtypes
        if subtype != primary_subtype
    ]

    explicit_subtypes = list(
        dict.fromkeys(
            explicit_group['impersonation_subtype']
            .dropna()
            .astype(str)
            .tolist()
        )
    )

    evidence_list = [
        str(value)
        for value in group.get(
            'evidence_text',
            pd.Series(dtype=str)
        ).dropna().unique()
        if str(value).strip()
    ]

    case_rows.append({
        'case_id': case_id,
        '접근사칭기관': access_subtype,
        '자동_주요사칭기관': primary_subtype,
        '보조사칭기관_목록': ' | '.join(
            secondary_subtypes
        ),
        '전체사칭후보_목록': ' | '.join(
            all_subtypes
        ),
        '명시적사칭기관_목록': ' | '.join(
            explicit_subtypes
        ),
        '사칭기관수': len(all_subtypes),
        '다중사칭여부': len(all_subtypes) > 1,
        '자동판단신뢰도': primary_confidence,
        '주요사칭_근거문장': primary_evidence,
        '전체사칭_근거문장': ' || '.join(
            evidence_list[:10]
        ),
        '검수필요여부': (
            primary_subtype == 'UNKNOWN'
            or len(explicit_subtypes) > 1
        ),
        '검수_주요사칭기관': '',
        '검수메모': ''
    })

case_impersonation_df = pd.DataFrame(case_rows)

# 원본 사기유형 연결
case_impersonation_df = case_impersonation_df.merge(
    type_map,
    on='case_id',
    how='left'
)

# 사건 전체 범인 대화 연결
case_text_column = next(
    (
        column
        for column in [
            'raw_offender_text',
            'normalized_offender_text',
            'raw_full_text',
            'normalized_full_text'
        ]
        if column in cases.columns
    ),
    None
)

if case_text_column:
    case_text_df = (
        cases[['case_id', case_text_column]]
        .drop_duplicates('case_id')
        .rename(
            columns={
                case_text_column: '사건전체_범인대화'
            }
        )
    )

    case_impersonation_df = (
        case_impersonation_df.merge(
            case_text_df,
            on='case_id',
            how='left'
        )
    )

# 사람이 보기 좋은 순서
front_columns = [
    'case_id',
    'supervised_target',
    '접근사칭기관',
    '자동_주요사칭기관',
    '명시적사칭기관_목록',
    '보조사칭기관_목록',
    '전체사칭후보_목록',
    '다중사칭여부',
    '자동판단신뢰도',
    '주요사칭_근거문장',
    '전체사칭_근거문장',
    '사건전체_범인대화',
    '검수필요여부',
    '검수_주요사칭기관',
    '검수메모'
]

front_columns = [
    column
    for column in front_columns
    if column in case_impersonation_df.columns
]

case_impersonation_df = case_impersonation_df[
    front_columns
]

display(case_impersonation_df.head(20))

case_impersonation_df.to_csv(
    TABLE_ROOT / '사건단위_사칭대상_검토표.csv',
    index=False,
    encoding='utf-8-sig'
)

print(
    '사건 수:',
    len(case_impersonation_df)
)

print(
    '검수 필요:',
    case_impersonation_df[
        '검수필요여부'
    ].sum()
)

print(
    '저장:',
    TABLE_ROOT / '사건단위_사칭대상_검토표.csv'
)

In [ ]:
# 사건 단위 사칭 대상 그래프

impersonation_name_map = {
    'BANK': '은행',
    'BUSINESS_CONTACT': '거래처',
    'CAPITAL_COMPANY': '캐피탈',
    'CARD_COMPANY': '카드회사',
    'COURT': '법원',
    'DELIVERY_COMPANY': '택배회사',
    'FINANCIAL_ASSOCIATION': '금융협회',
    'FSS': '금융감독원',
    'KIDNAPPING_CLAIM': '납치 주장',
    'LOAN_COMPANY': '대출회사',
    'PERSONAL_CONTACT': '지인·가족',
    'POLICE': '경찰',
    'POST_OFFICE': '우체국',
    'PROSECUTION': '검찰',
    'RECRUITER': '채용담당자',
    'SAVINGS_BANK': '저축은행',
    'TAX_AUTHORITY': '세무기관',
    'TELECOM': '통신회사',
    'UNKNOWN': '판단불가'
}

fraud_type_name_map = {
    'INSTITUTION_IMPERSONATION': '수사기관 사칭형',
    'LOAN_FRAUD': '대출사기형'
}

plot_case_df = case_impersonation_df.copy()

plot_case_df['사기유형'] = (
    plot_case_df['supervised_target']
    .map(fraud_type_name_map)
    .fillna(plot_case_df['supervised_target'])
)

plot_case_df['주요사칭기관'] = (
    plot_case_df['자동_주요사칭기관']
    .map(impersonation_name_map)
    .fillna(plot_case_df['자동_주요사칭기관'])
)

plot_case_df['접근사칭기관_한글'] = (
    plot_case_df['접근사칭기관']
    .map(impersonation_name_map)
    .fillna(plot_case_df['접근사칭기관'])
)


# 1. 사기유형별 주요 사칭기관 사건 비율
primary_ratio_df = pd.crosstab(
    plot_case_df['사기유형'],
    plot_case_df['주요사칭기관'],
    normalize='index'
) * 100

primary_count_df = pd.crosstab(
    plot_case_df['사기유형'],
    plot_case_df['주요사칭기관']
)

display(primary_count_df)
display(primary_ratio_df.round(1))

primary_count_df.to_csv(
    TABLE_ROOT / '사건단위_주요사칭기관_건수.csv',
    encoding='utf-8-sig'
)

primary_ratio_df.to_csv(
    TABLE_ROOT / '사건단위_주요사칭기관_비율.csv',
    encoding='utf-8-sig'
)

# 전체 사건에서 많이 등장한 상위 기관만 표시
top_institutions = (
    primary_count_df
    .sum(axis=0)
    .nlargest(10)
    .index
)

bar_df = (
    primary_ratio_df[top_institutions]
    .reset_index()
    .melt(
        id_vars='사기유형',
        var_name='주요사칭기관',
        value_name='사건비율'
    )
)

plt.figure(figsize=(15, 7))

ax = sns.barplot(
    data=bar_df,
    x='주요사칭기관',
    y='사건비율',
    hue='사기유형',
    palette=['#4c72b0', '#dd8452']
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=3,
        fontsize=8
    )

plt.title('보이스피싱 유형별 사건 단위 주요 사칭기관')
plt.xlabel('주요 사칭기관')
plt.ylabel('해당 유형 내 사건 비율(%)')
plt.xticks(rotation=25)
plt.legend(title='사기유형')
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '사건단위_주요사칭기관_막대그래프.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()


# 2. 주요 사칭기관 비율 히트맵
plt.figure(figsize=(15, 5))

sns.heatmap(
    primary_ratio_df,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar_kws={
        'label': '해당 유형 내 사건 비율(%)'
    }
)

plt.title('보이스피싱 유형별 사건 단위 주요 사칭기관')
plt.xlabel('주요 사칭기관')
plt.ylabel('사기유형')
plt.xticks(rotation=40, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '사건단위_주요사칭기관_히트맵.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()


# 3. 접근 사칭기관 → 주요 사칭기관 전환
transition_count_df = pd.crosstab(
    plot_case_df['접근사칭기관_한글'],
    plot_case_df['주요사칭기관']
)

# 너무 드문 기관을 제외하고 상위 기관만 표시
top_access = (
    transition_count_df
    .sum(axis=1)
    .nlargest(10)
    .index
)

top_primary = (
    transition_count_df
    .sum(axis=0)
    .nlargest(10)
    .index
)

transition_plot_df = transition_count_df.loc[
    top_access,
    top_primary
]

display(transition_plot_df)

transition_count_df.to_csv(
    TABLE_ROOT / '사건단위_접근기관_주요기관_전환표.csv',
    encoding='utf-8-sig'
)

plt.figure(figsize=(12, 8))

sns.heatmap(
    transition_plot_df,
    annot=True,
    fmt='g',
    cmap='YlOrRd'
)

plt.title('접근 사칭기관에서 주요 사칭기관으로의 전환')
plt.xlabel('주요 사칭기관')
plt.ylabel('처음 접근한 사칭기관')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '사건단위_사칭기관_전환히트맵.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()


# 4. 판단불가·다중 사칭 사건 비율
quality_plot_df = (
    plot_case_df
    .groupby('사기유형')
    .agg(
        전체사건수=('case_id', 'nunique'),
        판단불가사건수=(
            '자동_주요사칭기관',
            lambda values: values.eq('UNKNOWN').sum()
        ),
        다중사칭사건수=(
            '다중사칭여부',
            'sum'
        ),
        검수필요사건수=(
            '검수필요여부',
            'sum'
        )
    )
    .reset_index()
)

for column in [
    '판단불가사건수',
    '다중사칭사건수',
    '검수필요사건수'
]:
    quality_plot_df[
        column.replace('사건수', '비율')
    ] = (
        quality_plot_df[column]
        / quality_plot_df['전체사건수']
        * 100
    )

display(quality_plot_df)

quality_plot_df.to_csv(
    TABLE_ROOT / '사건단위_사칭판단_품질요약.csv',
    index=False,
    encoding='utf-8-sig'
)

quality_long_df = quality_plot_df.melt(
    id_vars='사기유형',
    value_vars=[
        '판단불가비율',
        '다중사칭비율',
        '검수필요비율'
    ],
    var_name='구분',
    value_name='사건비율'
)

plt.figure(figsize=(11, 6))

ax = sns.barplot(
    data=quality_long_df,
    x='구분',
    y='사건비율',
    hue='사기유형'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=3
    )

plt.title('사건 단위 사칭 판단 품질')
plt.xlabel('')
plt.ylabel('사건 비율(%)')
plt.legend(title='사기유형')
plt.tight_layout()

plt.savefig(
    FIGURE_ROOT / '사건단위_사칭판단_품질그래프.png',
    dpi=170,
    bbox_inches='tight'
)

plt.show()

print('사건 단위 사칭 대상 그래프 4개 저장 완료')

## 6. 심리전략 빈도·조합·사건 흐름

In [ ]:
# 11. 전략 분포와 사기유형별 전략
strategy=tables['vp_strategy_events']; strategy_col='strategy_type'
strategy_count=save_count(strategy,strategy_col,'14_심리전략_분포',TOP_N)
strategy_cross=event_crosstab(strategy,strategy_col,'15_사기유형_심리전략')
if not strategy_cross.empty:
    plt.figure(figsize=(14,4)); sns.heatmap(strategy_cross,annot=True,fmt='g',cmap='Oranges')
    plt.title('사기유형 × 심리전략'); plt.tight_layout(); plt.savefig(FIGURE_ROOT/'04_사기유형_심리전략.png',dpi=170); plt.show()

In [ ]:
# 12. 한 사건에서 함께 등장한 심리전략
strategy_binary=pd.crosstab(strategy['case_id'],strategy[strategy_col]).gt(0).astype(int)
cooccur=strategy_binary.T.dot(strategy_binary)
cooccur.to_csv(TABLE_ROOT/'16_심리전략_동시출현.csv',encoding='utf-8-sig')
display(cooccur)
plt.figure(figsize=(11,9)); sns.heatmap(cooccur,annot=True,fmt='g',cmap='YlOrRd')
plt.title('한 사건에서 함께 등장한 심리전략'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'05_심리전략_동시출현.png',dpi=170); plt.show()
dashboard=tables['dashboard_case_summary']
strategy_cols=[c for c in dashboard.columns if c.endswith('_count') and any(k in c for k in ['authority','behavior','benefit','fear','information','isolation','legitimacy','money','resistance','urgency'])]
if strategy_cols:
    corr=dashboard[strategy_cols].apply(pd.to_numeric,errors='coerce').corr(method='spearman')
    corr.to_csv(TABLE_ROOT/'17_심리전략_상관계수.csv',encoding='utf-8-sig')
    plt.figure(figsize=(11,9)); sns.heatmap(corr,annot=True,fmt='.2f',center=0,cmap='coolwarm')
    plt.title('사건별 심리전략 횟수의 Spearman 상관'); plt.tight_layout()
    plt.savefig(FIGURE_ROOT/'06_심리전략_상관.png',dpi=170); plt.show()

## 7. 금액 방향·용도·품질 분석

In [ ]:
# 13. 금액 상태·방향·용도 교차분석
amount=tables['vp_amount_events'].copy()
amount['amount_krw']=pd.to_numeric(amount['amount_krw'],errors='coerce')
amount['amount_10k_krw']=amount['amount_krw']/10000
status_direction=pd.crosstab(amount['amount_status'],amount['amount_direction'],margins=True)
direction_purpose=pd.crosstab(amount['amount_direction'],amount['amount_purpose'],margins=True)
display(status_direction); display(direction_purpose)
status_direction.to_csv(TABLE_ROOT/'18_금액상태_방향_교차표.csv',encoding='utf-8-sig')
direction_purpose.to_csv(TABLE_ROOT/'19_금액방향_용도_교차표.csv',encoding='utf-8-sig')
unknown_summary=pd.DataFrame([
 {'항목':'금액방향 미분류','건수':int(amount.amount_direction.eq('NO_DIRECTION').sum()),'비율':amount.amount_direction.eq('NO_DIRECTION').mean()},
 {'항목':'금액용도 미분류','건수':int(amount.amount_purpose.eq('UNKNOWN').sum()),'비율':amount.amount_purpose.eq('UNKNOWN').mean()}])
display(unknown_summary); unknown_summary.to_csv(TABLE_ROOT/'20_금액라벨_미분류율.csv',index=False,encoding='utf-8-sig')
direction_amount=amount.groupby('amount_direction')['amount_10k_krw'].agg(['count','median','mean','min','max']).round(2)
purpose_amount=amount.groupby('amount_purpose')['amount_10k_krw'].agg(['count','median','mean','min','max']).round(2)
display(direction_amount); display(purpose_amount)
direction_amount.to_csv(TABLE_ROOT/'21_금액방향별_금액만원.csv',encoding='utf-8-sig')
purpose_amount.to_csv(TABLE_ROOT/'22_금액용도별_금액만원.csv',encoding='utf-8-sig')

In [ ]:
# 14. 금액 그래프와 충돌·검토 대상
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.countplot(data=amount,x='amount_direction',order=amount.amount_direction.value_counts().index,ax=axes[0]); axes[0].tick_params(axis='x',rotation=25); axes[0].set_title('금액 방향')
top_purpose=amount.amount_purpose.value_counts().head(10).index
sns.countplot(data=amount[amount.amount_purpose.isin(top_purpose)],y='amount_purpose',order=top_purpose,ax=axes[1]); axes[1].set_title('금액 용도 상위 10개')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'07_금액방향_용도.png',dpi=170); plt.show()
upper=amount.amount_10k_krw.quantile(.99); plot_amount=amount[amount.amount_10k_krw.between(0,upper)]
plt.figure(figsize=(11,5)); sns.histplot(data=plot_amount,x='amount_10k_krw',hue='amount_direction',bins=30)
plt.xlabel('금액(만원)'); plt.title('금액 방향별 분포: 상위 1% 제외'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'08_금액방향별_만원분포.png',dpi=170); plt.show()
conflict_mask=(amount.amount_status.eq('REQUESTED') & ~amount.amount_direction.eq('REQUESTED_FROM_VICTIM')) | (amount.amount_direction.eq('REQUESTED_FROM_VICTIM') & ~amount.amount_status.eq('REQUESTED'))
review_cols=[c for c in ['amount_event_id','case_id','amount_krw','amount_status','amount_direction','amount_purpose','evidence_role','evidence_text','amount_direction_evidence','amount_direction_confidence'] if c in amount]
amount_conflicts=amount.loc[conflict_mask,review_cols].copy()
display(amount_conflicts.head(30)); amount_conflicts.to_csv(REVIEW_ROOT/'금액라벨_충돌_검토대상.csv',index=False,encoding='utf-8-sig')

## 8. 전체·부분 구간 분석

In [ ]:
# 15. 앞·중간·뒤·전체 구간의 표본과 길이
segment=tables['segment_detection_ml'].copy()
segment['text_length']=segment['window_text'].fillna('').str.len()
segment_summary=segment.groupby(['sample_scope','window_position','fraud_label']).agg(건수=('window_text','size'),텍스트길이_중앙값=('text_length','median'),텍스트길이_평균=('text_length','mean')).reset_index()
display(segment_summary); segment_summary.to_csv(TABLE_ROOT/'23_전체부분구간_분포.csv',index=False,encoding='utf-8-sig')
window=segment[segment.sample_scope.eq('WINDOW')]
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.countplot(data=window,x='window_position',hue='fraud_label',ax=axes[0]); axes[0].set_title('구간 위치별 표본 수')
sns.boxplot(data=window,x='window_position',y='text_length',hue='fraud_label',showfliers=False,ax=axes[1]); axes[1].set_title('구간 위치별 텍스트 길이')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'09_전체부분구간_분포.png',dpi=170); plt.show()

## 9. 사건 길이·발화·최초 위험행동

In [ ]:
# 16. 사건 단위 기술통계: 발화 비율은 대화 주도권의 확정값이 아닙니다.
case_source=dashboard if not dashboard.empty else cases
numeric_cols=[c for c in ['duration_sec','turn_count','speaker_count','offender_turn_count','victim_turn_count','unknown_turn_count','requested_action_count','strategy_diversity','first_risky_action_sec'] if c in case_source]
case_stats=case_source[numeric_cols].apply(pd.to_numeric,errors='coerce').describe(percentiles=[.25,.5,.75,.9,.95]).T.round(2) if numeric_cols else pd.DataFrame()
display(case_stats); case_stats.to_csv(TABLE_ROOT/'24_사건단위_기술통계.csv',encoding='utf-8-sig')
if {'first_risky_action_sec','duration_sec'}.issubset(case_source.columns):
    timing=case_source[['case_id','first_risky_action_sec','duration_sec']].copy()
    timing['최초위험행동_진행비율']=pd.to_numeric(timing.first_risky_action_sec,errors='coerce')/pd.to_numeric(timing.duration_sec,errors='coerce').replace(0,np.nan)
    timing=timing[timing['최초위험행동_진행비율'].between(0,1)]
    timing.to_csv(TABLE_ROOT/'25_최초위험행동_진행비율.csv',index=False,encoding='utf-8-sig')
    plt.figure(figsize=(9,5)); sns.histplot(timing['최초위험행동_진행비율'],bins=20)
    plt.xlabel('통화 진행 비율'); plt.title('최초 위험행동이 등장한 상대적 시점'); plt.tight_layout()
    plt.savefig(FIGURE_ROOT/'10_최초위험행동_시점.png',dpi=170); plt.show()

## 10. 최종 요약과 보고서 저장

In [ ]:
# 17. EDA 자동 요약
total_amount=len(amount); no_direction=int(amount.amount_direction.eq('NO_DIRECTION').sum()); unknown_purpose=int(amount.amount_purpose.eq('UNKNOWN').sum())
class_counts=det.fraud_label.value_counts().to_dict()
median_lengths=det.groupby('fraud_label').text_length.median().to_dict()
report=[
 '# 03-2 EDA 결과 요약','', '## 데이터 규모','',
 f"- 정상상담: {class_counts.get('LEGITIMATE_FINANCIAL_CALL',0):,}건",
 f"- 보이스피싱: {class_counts.get('VOICE_PHISHING',0):,}건",
 f"- 정상상담 텍스트 길이 중앙값: {median_lengths.get('LEGITIMATE_FINANCIAL_CALL',np.nan):,.0f}자",
 f"- 보이스피싱 텍스트 길이 중앙값: {median_lengths.get('VOICE_PHISHING',np.nan):,.0f}자",'',
 '## 주요 품질 이슈','',
 '- 정상상담과 보이스피싱은 출처와 원본 편집 방식이 달라 길이·문체 편향을 확인해야 합니다.',
 f'- 금액 방향 미분류: {no_direction:,}/{total_amount:,}건 ({no_direction/max(total_amount,1):.1%})',
 f'- 금액 용도 미분류: {unknown_purpose:,}/{total_amount:,}건 ({unknown_purpose/max(total_amount,1):.1%})',
 f'- 금액 상태·방향 충돌 검토 대상: {len(amount_conflicts):,}건','',
 '## 해석 기준','',
 '- 사칭·행동·전략·금액은 규칙 기반 SILVER 라벨이므로 탐색적 경향으로 해석합니다.',
 '- 건수와 함께 행 비율을 사용하며, 금액은 평균보다 중앙값을 우선 확인합니다.',
 '- 발화 수 차이는 대화 주도권의 확정 증거가 아닙니다.',
 '- 실제 피해 여부 정답이 없으므로 피해 발생 확률로 해석하지 않습니다.'
]
(REPORT_ROOT/'03_2_eda_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'03-2_v1','dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
          'tables':inventory.to_dict(orient='records'),'saved_csv':len(list(TABLE_ROOT.glob('*.csv'))),
          'saved_figures':len(list(FIGURE_ROOT.glob('*.png'))),'amount_conflict_rows':len(amount_conflicts)}
(REPORT_ROOT/'03_2_eda_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('분석표:',len(list(TABLE_ROOT.glob('*.csv'))),'개')
print('그래프:',len(list(FIGURE_ROOT.glob('*.png'))),'개')
print('03-2 EDA 정상 완료:',OUTPUT_ROOT)

---

# PART B — 해석 가능한 특징 분석

원본: `03_3_interpretable_feature_analysis\03_3_interpretable_feature_analysis_v1.ipynb`


# PART B. v5 해석 가능한 특징 기반 지도·비지도 분석

정상 금융상담과 보이스피싱에 동일한 규칙을 적용하여 금전요구·개인정보요구·강압·긴급성·위협·비밀유지 등의 특징을 추출합니다.

- 지도학습: Logistic Regression, Linear SVM, Random Forest, Extra Trees 비교
- 비지도학습: K-means로 정상·사기 혼합 군집 및 보이스피싱 내부 전술 군집 탐색
- 원문 길이·출처·파일명·금융주제는 모델 입력에서 제외
- 횟수 대신 1,000자당 표현 밀도와 존재 여부 사용


In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn scipy seaborn matplotlib koreanize-matplotlib joblib openpyxl

In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import json, re, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, average_precision_score,
    classification_report, confusion_matrix, normalized_mutual_info_score,
    precision_recall_fscore_support, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
warnings.filterwarnings('ignore')
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 설정값

In [ ]:
# 2. 경로와 공통 설정
DRIVE_ROOT=Path('/content/drive/MyDrive')
PROJECT_ROOT=DRIVE_ROOT/'보이스피싱_분석'
DATASET_ROOT=PROJECT_ROOT/'구축 데이터셋_v4'
ML_ROOT=DATASET_ROOT/'02_ml_tables'
OUTPUT_ROOT=PROJECT_ROOT/'머신러닝_분석결과_v5' / '02_해석형특징'
TABLE_ROOT=OUTPUT_ROOT/'01_분석표'
FIGURE_ROOT=OUTPUT_ROOT/'02_그래프'
MODEL_ROOT=OUTPUT_ROOT/'03_모델'
PRED_ROOT=OUTPUT_ROOT/'04_예측_군집결과'
REPORT_ROOT=OUTPUT_ROOT/'05_보고서'
for folder in [TABLE_ROOT,FIGURE_ROOT,MODEL_ROOT,PRED_ROOT,REPORT_ROOT]: folder.mkdir(parents=True,exist_ok=True)
SEED=42
TEST_RATIO=.20
DEV_RATIO=.20
FRAUD='VOICE_PHISHING'
NORMAL='LEGITIMATE_FINANCIAL_CALL'
assert ML_ROOT.exists(),f'구축 데이터셋_v4 경로를 확인하세요: {ML_ROOT}'
print('입력:',DATASET_ROOT); print('출력:',OUTPUT_ROOT)

## 2. 데이터 불러오기

In [ ]:
# 3. 정상·사기와 사기유형 테이블
def read_table(folder,name):
    pq=folder/f'{name}.parquet'; csv=folder/f'{name}.csv'
    if pq.exists(): return pd.read_parquet(pq)
    assert csv.exists(),f'{name}을 찾지 못했습니다.'
    return pd.read_csv(csv,encoding='utf-8-sig')
det=read_table(ML_ROOT,'fraud_detection_ml')
type_df=read_table(ML_ROOT,'fraud_type_ml')
required={'conversation_id','group_id','fraud_label','model_input_text','original_split'}
assert required.issubset(det.columns),f'필수 컬럼 누락: {required-set(det.columns)}'
det=det.dropna(subset=['group_id','fraud_label','model_input_text']).copy()
det['group_id']=det['group_id'].astype(str)
display(det.fraud_label.value_counts().rename_axis('구분').reset_index(name='건수'))
print('분석 대상:',len(det),'건')

## 3. 동일 규칙으로 해석형 특징 추출

아래 규칙은 정답 라벨을 보지 않고 정상상담과 보이스피싱 전체에 동일하게 적용합니다. 자동 추출 결과이므로 `SILVER` 특징입니다.

In [ ]:
# 4. 행동·심리·말투 특징 규칙
FEATURE_RULES={
 'money_request':r'송금|이체|입금|납부|지불|결제|돈.{0,8}(보내|내|줘|준비)|금액.{0,8}(보내|입금)',
 'transfer_cash':r'계좌.{0,8}(이체|송금)|현금.{0,8}(인출|찾|전달)|ATM|씨디기|CD기',
 'fee_tax_deposit':r'수수료|선입금|보증금|예치금|공탁금|세금|과태료|벌금|인지대',
 'account_info':r'계좌번호|잔액|통장|카드번호|금융거래|거래내역',
 'personal_info':r'주민번호|주민등록|생년월일|신분증|주소|개인정보|명의',
 'auth_code':r'인증번호|비밀번호|보안카드|OTP|일회용.{0,3}비밀번호',
 'app_remote':r'앱.{0,8}(설치|깔)|어플.{0,8}(설치|깔)|원격.{0,8}(접속|제어)|팀뷰어|퀵서포트',
 'command_pressure':r'하세요|하셔야|해야 합니다|따라 하|지금.{0,8}(가|하|보내|이체)|시키는 대로|말씀드린 대로',
 'urgency_pressure':r'지금 당장|즉시|긴급|오늘 안|시간이 없|빨리|지체하면|마감|몇 분 안',
 'fear_threat':r'체포|구속|압류|범죄|수배|처벌|고소|고발|피해를 입|큰일|위험|납치',
 'isolation_secrecy':r'비밀|말하지 마|알리면 안|누구에게도|혼자만|통화.{0,8}(끊지|유지)|전화.{0,8}(끊지|받지)',
 'authority_trust':r'검찰|검사|경찰|수사관|법원|금융감독원|금감원|은행 본점|정부기관|공문|사건번호',
 'resistance_handling':r'의심|못 믿|확인해 보|그게 아니라|걱정하지|안심|오해|설명드리',
 'benefit_offer':r'대출.{0,10}(승인|가능|해드리)|환급|돌려드리|지원금|혜택|저금리|금리.{0,8}(낮|인하)|한도.{0,8}(상향|증액)',
}
FEATURE_KO={
 'money_request':'금전·송금요구','transfer_cash':'송금·현금행동','fee_tax_deposit':'수수료·세금·보증금',
 'account_info':'계좌정보 관련표현','personal_info':'개인정보 관련표현','auth_code':'인증정보 관련표현',
 'app_remote':'앱설치·원격접속','command_pressure':'명령·강압','urgency_pressure':'긴급성·시간압박',
 'fear_threat':'공포·위협','isolation_secrecy':'고립·비밀유지','authority_trust':'권위·신뢰형성',
 'resistance_handling':'의심·저항대응','benefit_offer':'이익·혜택제안'}
compiled_rules={name:re.compile(pattern,re.I) for name,pattern in FEATURE_RULES.items()}
print('추출 특징:',len(compiled_rules),'개')

In [ ]:
# 5. 통화 길이 대신 1,000자당 표현 밀도와 존재 여부 생성
def extract_features(text):
    text=re.sub(r'\s+',' ',str(text or '')).strip()
    denominator=max(len(text),1)
    row={}
    present=0
    for name,pattern in compiled_rules.items():
        count=len(pattern.findall(text))
        row[f'{name}_rate_1k']=count/denominator*1000
        row[f'{name}_flag']=int(count>0)
        present+=int(count>0)
    row['risk_feature_diversity_ratio']=present/len(compiled_rules)
    return row
feature_values=pd.DataFrame(det.model_input_text.map(extract_features).tolist(),index=det.index)
feature_df=pd.concat([det[['conversation_id','group_id','fraud_label','original_split']].copy(),feature_values],axis=1)
feature_cols=list(feature_values.columns)
assert not feature_values.isna().any().any()
assert not any(x in feature_cols for x in ['text_length','source_group','financial_topic','original_split'])
feature_df.to_csv(TABLE_ROOT/'해석형_특징_데이터.csv',index=False,encoding='utf-8-sig')
display(feature_df.head()); print('모델 입력 특징:',len(feature_cols),'개')

## 4. 특징별 정상·사기 차이와 통계 검정

In [ ]:
# 6. 밀도 중앙값·존재율·Mann-Whitney 검정
def bh_fdr(p_values):
    p=np.asarray(p_values,float); order=np.argsort(p); ranked=p[order]*len(p)/(np.arange(len(p))+1)
    ranked=np.minimum.accumulate(ranked[::-1])[::-1]; result=np.empty_like(ranked); result[order]=np.clip(ranked,0,1); return result
stats=[]
for name in FEATURE_RULES:
    rate=f'{name}_rate_1k'; flag=f'{name}_flag'
    normal=feature_df.loc[feature_df.fraud_label.eq(NORMAL),rate]
    fraud=feature_df.loc[feature_df.fraud_label.eq(FRAUD),rate]
    _,p=mannwhitneyu(fraud,normal,alternative='two-sided')
    flag_table=pd.crosstab(feature_df.fraud_label,feature_df[flag])
    flag_p=chi2_contingency(flag_table)[1] if flag_table.shape==(2,2) else 1.0
    stats.append({'특징코드':name,'특징':FEATURE_KO[name],
      '정상_1000자당평균':normal.mean(),'사기_1000자당평균':fraud.mean(),
      '정상_존재율':feature_df.loc[feature_df.fraud_label.eq(NORMAL),flag].mean(),
      '사기_존재율':feature_df.loc[feature_df.fraud_label.eq(FRAUD),flag].mean(),
      '평균밀도차이':fraud.mean()-normal.mean(),'밀도_p_value':p,'존재여부_카이제곱_p_value':flag_p})
stats_df=pd.DataFrame(stats)
stats_df['밀도_fdr_p_value']=bh_fdr(stats_df['밀도_p_value'])
stats_df['존재여부_카이제곱_fdr_p_value']=bh_fdr(stats_df['존재여부_카이제곱_p_value'])
stats_df=stats_df.sort_values('평균밀도차이',ascending=False)
display(stats_df); stats_df.to_csv(TABLE_ROOT/'특징별_정상사기_차이.csv',index=False,encoding='utf-8-sig')
plot_df=stats_df.sort_values('평균밀도차이')
plt.figure(figsize=(10,7)); sns.barplot(data=plot_df,y='특징',x='평균밀도차이',color='#377eb8')
plt.axvline(0,color='black',lw=1); plt.xlabel('사기 - 정상: 1,000자당 평균 출현 차이'); plt.title('행동·심리 표현 밀도 차이')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'01_특징밀도_정상사기차이.png',dpi=170); plt.show()

## 5. 원본 통화 단위 학습·검증·테스트 분리

In [ ]:
# 7. 정상상담 공식 Validation 보존, 보이스피싱은 원본 file_id 그룹 분리
def random_group_map(groups):
    groups=np.array(sorted(pd.Series(groups).dropna().astype(str).unique()))
    train_dev,test=train_test_split(groups,test_size=TEST_RATIO,random_state=SEED)
    train,dev=train_test_split(train_dev,test_size=DEV_RATIO,random_state=SEED)
    out={g:'TRAIN' for g in train}; out.update({g:'DEV' for g in dev}); out.update({g:'TEST' for g in test}); return out
feature_df['ml_split']=''
normal_mask=feature_df.fraud_label.eq(NORMAL)
official=feature_df.original_split.fillna('').astype(str).str.upper()
normal_test=normal_mask & official.str.contains('VALID|TEST')
if normal_test.any():
    normal_pool=normal_mask & ~normal_test
    groups=feature_df.loc[normal_pool,'group_id'].unique(); tr_group,dv_group=train_test_split(groups,test_size=DEV_RATIO,random_state=SEED)
    feature_df.loc[normal_pool & feature_df.group_id.isin(tr_group),'ml_split']='TRAIN'
    feature_df.loc[normal_pool & feature_df.group_id.isin(dv_group),'ml_split']='DEV'
    feature_df.loc[normal_test,'ml_split']='TEST'
else:
    normal_map=random_group_map(feature_df.loc[normal_mask,'group_id']); feature_df.loc[normal_mask,'ml_split']=feature_df.loc[normal_mask,'group_id'].map(normal_map)
fraud_map=random_group_map(feature_df.loc[~normal_mask,'group_id']); feature_df.loc[~normal_mask,'ml_split']=feature_df.loc[~normal_mask,'group_id'].map(fraud_map)
assert not feature_df.ml_split.eq('').any()
assert feature_df.groupby('group_id').ml_split.nunique().max()==1,'원본 통화 누출이 있습니다.'
split_summary=feature_df.groupby(['ml_split','fraud_label']).size().reset_index(name='건수')
display(split_summary); split_summary.to_csv(TABLE_ROOT/'학습검증테스트_분포.csv',index=False,encoding='utf-8-sig')

## 6. 해석형 특징 지도학습

In [ ]:
# 8. 후보 알고리즘
def candidates():
    return {
     'Dummy':DummyClassifier(strategy='prior'),
     'Logistic_Regression':Pipeline([('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))]),
     'Linear_SVM':Pipeline([('scale',StandardScaler()),('model',LinearSVC(class_weight='balanced',random_state=SEED))]),
     'Random_Forest':RandomForestClassifier(n_estimators=500,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1),
     'Extra_Trees':ExtraTreesClassifier(n_estimators=500,min_samples_leaf=3,class_weight='balanced',random_state=SEED,n_jobs=-1)}
def score_values(model,x):
    idx=list(model.classes_).index(FRAUD)
    if hasattr(model,'predict_proba'): return model.predict_proba(x)[:,idx]
    score=model.decision_function(x); return score if idx==1 else -score
def metrics(y,pred,score):
    p,r,f1,_=precision_recall_fscore_support(y,pred,average='binary',pos_label=FRAUD,zero_division=0)
    return {'accuracy':accuracy_score(y,pred),'precision':p,'recall':r,'f1':f1,
            'pr_auc':average_precision_score((np.asarray(y)==FRAUD).astype(int),score)}
train=feature_df[feature_df.ml_split.eq('TRAIN')]; dev=feature_df[feature_df.ml_split.eq('DEV')]; test=feature_df[feature_df.ml_split.eq('TEST')]
compare_rows=[]
for name,model in candidates().items():
    model.fit(train[feature_cols],train.fraud_label); pred=model.predict(dev[feature_cols]); score=score_values(model,dev[feature_cols])
    compare_rows.append({'모델':name,**metrics(dev.fraud_label,pred,score)}); print(name,'완료')
compare_df=pd.DataFrame(compare_rows).sort_values(['pr_auc','f1'],ascending=False).reset_index(drop=True)
display(compare_df); compare_df.to_csv(TABLE_ROOT/'지도학습_모델비교_검증.csv',index=False,encoding='utf-8-sig')

In [ ]:
# 9. 선정 모델 최종 테스트 한 번
best_name=compare_df.iloc[0]['모델']; best_model=clone(candidates()[best_name])
train_dev=feature_df[feature_df.ml_split.isin(['TRAIN','DEV'])]
best_model.fit(train_dev[feature_cols],train_dev.fraud_label)
test_pred=best_model.predict(test[feature_cols]); test_score=score_values(best_model,test[feature_cols])
test_metrics=metrics(test.fraud_label,test_pred,test_score)
display(pd.DataFrame([{'모델':best_name,**test_metrics}]))
print(classification_report(test.fraud_label,test_pred,zero_division=0))
pred_df=test[['conversation_id','group_id','fraud_label']].copy(); pred_df['예측']=test_pred; pred_df['보이스피싱점수']=test_score; pred_df['정답여부']=pred_df.fraud_label.eq(test_pred)
pred_df.to_csv(PRED_ROOT/'해석형특징_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
joblib.dump({'model':best_model,'feature_columns':feature_cols,'feature_rules':FEATURE_RULES},MODEL_ROOT/'interpretable_fraud_model.joblib')
cm=confusion_matrix(test.fraud_label,test_pred,labels=[NORMAL,FRAUD])
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['정상','보이스피싱'],yticklabels=['정상','보이스피싱'])
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('해석형 특징 모델 최종 테스트'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'02_해석형모델_혼동행렬.png',dpi=170); plt.show()

In [ ]:
# 10. 최종 모델 변수 중요도
estimator=best_model.named_steps['model'] if isinstance(best_model,Pipeline) else best_model
if hasattr(estimator,'coef_'):
    importance=np.asarray(estimator.coef_).ravel()
elif hasattr(estimator,'feature_importances_'):
    importance=np.asarray(estimator.feature_importances_)
else:
    importance=np.zeros(len(feature_cols))
importance_df=pd.DataFrame({'특징코드':feature_cols,'중요도':importance})
importance_df['특징']=importance_df.특징코드.map(lambda x:FEATURE_KO.get(x.replace('_rate_1k','').replace('_flag',''),x))
importance_df['형태']=np.where(importance_df.특징코드.str.endswith('_flag'),'존재여부',np.where(importance_df.특징코드.str.endswith('_rate_1k'),'1,000자당밀도','다양성'))
importance_df['절대중요도']=importance_df.중요도.abs(); importance_df=importance_df.sort_values('절대중요도',ascending=False)
display(importance_df.head(20)); importance_df.to_csv(TABLE_ROOT/'최종모델_변수중요도.csv',index=False,encoding='utf-8-sig')
top=importance_df.head(20).sort_values('중요도')
plt.figure(figsize=(10,8)); sns.barplot(data=top,y='특징코드',x='중요도',color='#4c72b0')
plt.axvline(0,color='black',lw=1); plt.title(f'{best_name} 변수 중요도'); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'03_최종모델_변수중요도.png',dpi=170); plt.show()

## 7. 비지도 K-means: 정상·사기 혼합 군집

K-means는 정상·사기 정답을 보지 않고 특징이 비슷한 통화를 묶습니다. 군집 생성 후에만 정답과의 대응을 확인합니다.

In [ ]:
# 11. 다수 클래스 영향을 줄이기 위한 1:1 표본
fraud_part=feature_df[feature_df.fraud_label.eq(FRAUD)]
normal_part=feature_df[feature_df.fraud_label.eq(NORMAL)].sample(n=len(fraud_part),random_state=SEED)
cluster_input=pd.concat([fraud_part,normal_part]).sample(frac=1,random_state=SEED).reset_index(drop=True)
scaler=StandardScaler(); cluster_x=scaler.fit_transform(cluster_input[feature_cols])
cluster_rows=[]; cluster_models={}
for k in range(2,9):
    model=KMeans(n_clusters=k,n_init=30,random_state=SEED); label=model.fit_predict(cluster_x)
    model2=KMeans(n_clusters=k,n_init=30,random_state=SEED+1); label2=model2.fit_predict(cluster_x)
    cluster_rows.append({'군집수':k,'silhouette':silhouette_score(cluster_x,label),'seed_stability_ari':adjusted_rand_score(label,label2)})
    cluster_models[k]=(model,label)
k_compare=pd.DataFrame(cluster_rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
display(k_compare); k_compare.to_csv(TABLE_ROOT/'KMeans_정상사기_군집수비교.csv',index=False,encoding='utf-8-sig')
best_k=int(k_compare.iloc[0].군집수); kmeans,mixed_labels=cluster_models[best_k]
cluster_input['군집ID']=mixed_labels
alignment=pd.crosstab(cluster_input.군집ID,cluster_input.fraud_label,margins=True)
ari=adjusted_rand_score(cluster_input.fraud_label,mixed_labels); nmi=normalized_mutual_info_score(cluster_input.fraud_label,mixed_labels)
display(alignment); print('정답과 군집의 ARI:',round(ari,4),'NMI:',round(nmi,4))
alignment.to_csv(TABLE_ROOT/'KMeans_군집_정상사기대응.csv',encoding='utf-8-sig')
cluster_input.to_csv(PRED_ROOT/'KMeans_정상사기_군집결과.csv',index=False,encoding='utf-8-sig')

In [ ]:
# 12. 군집 시각화와 군집별 특징 프로필
pca=PCA(n_components=2,random_state=SEED); xy=pca.fit_transform(cluster_x)
plot_df=pd.DataFrame({'PCA1':xy[:,0],'PCA2':xy[:,1],'군집ID':mixed_labels,'실제구분':cluster_input.fraud_label})
fig,axes=plt.subplots(1,2,figsize=(15,6))
sns.scatterplot(data=plot_df,x='PCA1',y='PCA2',hue='군집ID',palette='tab10',alpha=.6,ax=axes[0]); axes[0].set_title('정답을 사용하지 않은 K-means 군집')
sns.scatterplot(data=plot_df,x='PCA1',y='PCA2',hue='실제구분',alpha=.6,ax=axes[1]); axes[1].set_title('같은 좌표의 실제 정상·사기')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'04_KMeans_정상사기_군집.png',dpi=170); plt.show()
profile=cluster_input.groupby('군집ID')[feature_cols].mean().T
profile.to_csv(TABLE_ROOT/'KMeans_군집별_특징프로필.csv',encoding='utf-8-sig')
display(profile.head(30))
joblib.dump({'scaler':scaler,'model':kmeans,'feature_columns':feature_cols},MODEL_ROOT/'kmeans_mixed_calls.joblib')

## 8. 비지도 K-means: 보이스피싱 내부 전술 유형

In [ ]:
# 13. 보이스피싱만 사용하여 행동·심리 패턴이 유사한 사건 묶기
fraud_cluster=feature_df[feature_df.fraud_label.eq(FRAUD)].copy().reset_index(drop=True)
fraud_x=StandardScaler().fit_transform(fraud_cluster[feature_cols])
rows=[]; models={}
for k in range(2,7):
    model=KMeans(n_clusters=k,n_init=30,random_state=SEED); labels=model.fit_predict(fraud_x)
    other=KMeans(n_clusters=k,n_init=30,random_state=SEED+1).fit_predict(fraud_x)
    rows.append({'군집수':k,'silhouette':silhouette_score(fraud_x,labels),'seed_stability_ari':adjusted_rand_score(labels,other)})
    models[k]=(model,labels)
fraud_k_compare=pd.DataFrame(rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
best_fraud_k=int(fraud_k_compare.iloc[0].군집수); fraud_model,fraud_labels=models[best_fraud_k]
fraud_cluster['전술군집ID']=fraud_labels
fraud_cluster['case_id']=fraud_cluster.conversation_id.str.replace(r'^fraud_','',regex=True)
type_map=type_df[['case_id','supervised_target']].drop_duplicates('case_id')
fraud_cluster=fraud_cluster.merge(type_map,on='case_id',how='left')
type_alignment=pd.crosstab(fraud_cluster.전술군집ID,fraud_cluster.supervised_target.fillna('MIXED_UNKNOWN'),margins=True)
profile=fraud_cluster.groupby('전술군집ID')[feature_cols].mean().T
display(fraud_k_compare); display(type_alignment); display(profile)
fraud_k_compare.to_csv(TABLE_ROOT/'KMeans_보이스피싱_군집수비교.csv',index=False,encoding='utf-8-sig')
type_alignment.to_csv(TABLE_ROOT/'KMeans_전술군집_기존유형대응.csv',encoding='utf-8-sig')
profile.to_csv(TABLE_ROOT/'KMeans_전술군집_특징프로필.csv',encoding='utf-8-sig')
fraud_cluster.to_csv(PRED_ROOT/'KMeans_보이스피싱_전술군집결과.csv',index=False,encoding='utf-8-sig')
joblib.dump({'model':fraud_model,'feature_columns':feature_cols},MODEL_ROOT/'kmeans_fraud_tactics.joblib')

## 9. 결과 요약과 한계

In [ ]:
# 14. 자동 보고서와 실행 기록
top_features=importance_df.head(8).특징코드.tolist()
report=[
 '# 03-3 해석형 특징 기반 분석 결과','', '## 지도학습','',
 f'- 최종 모델: {best_name}',f"- 최종 F1: {test_metrics['f1']:.4f}",f"- 최종 Recall: {test_metrics['recall']:.4f}",
 f"- 최종 PR-AUC: {test_metrics['pr_auc']:.4f}",f"- 주요 특징: {', '.join(top_features)}",'',
 '## 비지도학습','',f'- 정상·사기 혼합 K-means 최적 k: {best_k}',f'- 정답과 군집의 ARI: {ari:.4f}',f'- 정답과 군집의 NMI: {nmi:.4f}',
 f'- 보이스피싱 내부 전술 군집수: {best_fraud_k}','',
 '## 해석 시 주의','',
 '- 모든 특징은 정상·사기에 동일한 규칙을 적용했지만 사람이 확정한 GOLD가 아닌 SILVER 특징입니다.',
 '- 원문 길이와 출처 컬럼은 모델에서 제외했지만, 편집된 보이스피싱의 위험표현 밀도가 높다는 편향은 남을 수 있습니다.',
 '- K-means 군집은 사전 정답 없이 만든 유사 패턴이며 정상·사기 분류 정답이 아닙니다.',
 '- 군집 이름은 특징 프로필과 대표 원문을 확인한 뒤 사람이 부여해야 합니다.',
 '- 현재 결과는 두 공개 코퍼스 분석이며 실제 서비스 일반화 성능은 외부 데이터로 검증해야 합니다.'
]
(REPORT_ROOT/'03_3_interpretable_feature_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'03-3_v1','seed':SEED,'dataset_root':str(DATASET_ROOT),'feature_count':len(feature_cols),
          'best_supervised_model':best_name,'test_metrics':test_metrics,'mixed_kmeans_k':best_k,
          'mixed_kmeans_ari':ari,'mixed_kmeans_nmi':nmi,'fraud_tactic_k':best_fraud_k,
          'group_leakage_check':True,'excluded_features':['text_length','source_group','financial_topic','original_split','file_name']}
(REPORT_ROOT/'03_3_run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
assert feature_df.groupby('group_id').ml_split.nunique().max()==1
assert len(list(MODEL_ROOT.glob('*.joblib')))==3
print('03-3 정상 완료:',OUTPUT_ROOT)

---

# PART C — 머신러닝 학습·평가

원본: `03_ml_training_evaluation_v4.ipynb`


# PART C. v5 머신러닝 분석·학습·평가

`구축 데이터셋_v4`를 이용해 정상 금융상담과 보이스피싱을 비교하고, 보이스피싱 유형 분류·구간 탐지·유사 사건 군집화를 수행합니다.

v4 추가 사항
- 정상상담과 보이스피싱의 텍스트 길이 차이 점검
- 학습 데이터 1:3과 1:1 비율 비교
- 긴 정상상담을 짧은 구간으로 맞춘 길이 보정 실험
- 금액의 방향(피해자에게 제시/피해자에게 요구/단순 언급)과 용도 EDA
- 모델 선택은 검증 데이터로 하고 최종 테스트는 한 번만 실행


In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn seaborn matplotlib koreanize-matplotlib joblib openpyxl

In [ ]:
# 1. 라이브러리 불러오기 및 Google Drive 연결
from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib, json, re, unicodedata, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, adjusted_rand_score, average_precision_score,
    classification_report, confusion_matrix, precision_recall_fscore_support, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
warnings.filterwarnings('ignore')
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 설정값

In [ ]:
# 2. 경로와 공통 설정
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v4'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
OUTPUT_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v5' / '03_ML학습평가'
KOREAN_ROOT = OUTPUT_ROOT / '00_한글_확인용'
EDA_ROOT = OUTPUT_ROOT / '01_EDA'
SPLIT_ROOT = OUTPUT_ROOT / '02_데이터분리'
MODEL_ROOT = OUTPUT_ROOT / '03_모델'
PRED_ROOT = OUTPUT_ROOT / '04_예측결과'
REPORT_ROOT = OUTPUT_ROOT / '05_보고서'
for folder in [KOREAN_ROOT, EDA_ROOT, SPLIT_ROOT, MODEL_ROOT, PRED_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
SEED = 42
TEST_RATIO = 0.20
DEV_RATIO = 0.20
MAX_FEATURES = 50000
MIN_TEXT_LENGTH = 10
assert ML_ROOT.exists(), f'1단계 결과 경로를 확인하세요: {ML_ROOT}'
print('입력:', DATASET_ROOT)
print('출력:', OUTPUT_ROOT)

## 2. 분석 시나리오

In [ ]:
# 3. 분석 목적과 평가 지표
scenario_df = pd.DataFrame([
    ['정상상담 vs 보이스피싱', '이진 분류', 'Recall·F1·PR-AUC'],
    ['보이스피싱 유형', '대출사기형 vs 수사기관사칭형', 'Macro F1·혼동행렬'],
    ['전체·부분 구간 탐지', '대화 어느 구간에서도 탐지 가능한지 확인', '구간별 Recall·F1·PR-AUC'],
    ['유사 사건', '정답 없이 비슷한 사건 묶기', 'Silhouette·seed 안정성'],
    ['길이·비율 민감도', '출처와 길이 차이에 의한 편향 확인', '검증 Recall·F1·PR-AUC'],
], columns=['시나리오','목적','평가지표'])
display(scenario_df)
scenario_df.to_csv(REPORT_ROOT/'분석_시나리오.csv', index=False, encoding='utf-8-sig')
print('실제 피해 여부 정답이 없으므로 피해 발생 확률은 학습하지 않습니다.')

## 3. 데이터 불러오기와 전처리

In [ ]:
# 4. 1단계 데이터 불러오기
def read_table(folder, name):
    parquet_path = folder / f'{name}.parquet'
    csv_path = folder / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    assert csv_path.exists(), f'테이블을 찾지 못했습니다: {name}'
    return pd.read_csv(csv_path, encoding='utf-8-sig')
detection_df = read_table(ML_ROOT, 'fraud_detection_ml')
type_df = read_table(ML_ROOT, 'fraud_type_ml')
segment_df = read_table(ML_ROOT, 'segment_detection_ml')
cluster_df = read_table(ML_ROOT, 'case_clustering_ml')
amount_df = read_table(STANDARD_ROOT, 'vp_amount_events')
summary_df = pd.DataFrame([
    ['정상·사기', len(detection_df), len(detection_df.columns)],
    ['사기유형', len(type_df), len(type_df.columns)],
    ['전체·부분구간', len(segment_df), len(segment_df.columns)],
    ['사건군집', len(cluster_df), len(cluster_df.columns)],
    ['금액이벤트', len(amount_df), len(amount_df.columns)],
], columns=['데이터','행수','컬럼수'])
display(summary_df)
required_amount = {'amount_krw','amount_status','amount_direction','amount_purpose'}
assert required_amount.issubset(amount_df.columns), f'v3 금액 컬럼 누락: {required_amount-set(amount_df.columns)}'

In [ ]:
# 5. 텍스트 정리: 원문은 보존하고 clean_text를 별도로 만듭니다.
def clean_text(text):
    text = unicodedata.normalize('NFKC', str(text or '')).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'(?m)^\s*(tx|rx|화자\s*\d*|범인|피해자)\s*[:：]\s*', ' ', text)
    text = re.sub(r'[*#xX]{2,}', ' 마스킹 ', text)
    text = re.sub(r'\b\d{2,}\b', ' 숫자 ', text)
    return re.sub(r'\s+', ' ', text).strip()
def prepare(df, text_col):
    result = df.copy()
    result['clean_text'] = result[text_col].fillna('').map(clean_text)
    result = result[result['clean_text'].str.len() >= MIN_TEXT_LENGTH].copy()
    result['text_hash'] = result['clean_text'].map(lambda x: hashlib.sha256(x.encode()).hexdigest())
    return result.reset_index(drop=True)
def remove_conflicts_and_duplicates(df, label):
    conflict = df.groupby('text_hash')[label].nunique()
    conflict_hashes = set(conflict[conflict > 1].index)
    result = df[~df['text_hash'].isin(conflict_hashes)].copy()
    duplicate_count = int(result.duplicated([label,'text_hash']).sum())
    return result.drop_duplicates([label,'text_hash']).reset_index(drop=True), len(conflict_hashes), duplicate_count
detection_df = prepare(detection_df, 'model_input_text')
type_df = prepare(type_df, 'model_input_text')
segment_df = prepare(segment_df, 'window_text')
cluster_df = prepare(cluster_df, 'model_input_text')
detection_df, d_conflicts, d_duplicates = remove_conflicts_and_duplicates(detection_df, 'fraud_label')
type_df, t_conflicts, t_duplicates = remove_conflicts_and_duplicates(type_df, 'supervised_target')
amount_df['amount_krw'] = pd.to_numeric(amount_df['amount_krw'], errors='coerce')
amount_df['amount_10k_krw'] = amount_df['amount_krw'] / 10000
amount_df['log_amount_10k_krw'] = np.log1p(amount_df['amount_10k_krw'].clip(lower=0))
display(pd.DataFrame([['정상·사기',d_conflicts,d_duplicates,len(detection_df)],
                      ['사기유형',t_conflicts,t_duplicates,len(type_df)]],
                     columns=['데이터','라벨충돌해시','정확중복제외','최종행수']))

## 4. 사람이 확인하는 한글 컬럼 데이터

In [ ]:
# 6. 분석 내부는 영문 컬럼, 확인용 CSV는 한글 컬럼으로 저장합니다.
column_ko = {
 'conversation_id':'대화ID','case_id':'사건ID','file_id':'원본파일ID','group_id':'분리그룹ID',
 'fraud_label':'사기여부','supervised_target':'보이스피싱유형','source_group':'데이터출처',
 'sample_scope':'표본범위','window_position':'구간위치','model_input_text':'모델입력원문',
 'window_text':'구간원문','clean_text':'정제텍스트','original_split':'원본데이터구분',
 'amount_krw':'금액_원','amount_10k_krw':'금액_만원','amount_text':'금액원문',
 'amount_status':'금액상태','amount_direction':'금액방향','amount_purpose':'금액용도',
 'amount_direction_evidence':'금액방향_근거문장','amount_direction_confidence':'금액방향_신뢰도',
 'evidence_text':'근거발화','evidence_role':'근거화자역할'
}
def save_korean(df, filename):
    result = df.rename(columns=column_ko)
    result.to_csv(KOREAN_ROOT/filename, index=False, encoding='utf-8-sig')
    return result
save_korean(detection_df, '정상상담_보이스피싱_분류데이터.csv')
save_korean(type_df, '보이스피싱_유형분류데이터.csv')
save_korean(segment_df, '전체_부분구간_탐지데이터.csv')
save_korean(cluster_df, '유사사건_군집데이터.csv')
amount_ko = save_korean(amount_df, '금액이벤트_방향_용도_만원.csv')
display(amount_ko.head(5))
print('한글 확인용 CSV 저장:', KOREAN_ROOT)

## 5. EDA와 편향 점검

In [ ]:
# 7. 클래스·텍스트 길이·금액 방향 확인
detection_df['text_length'] = detection_df['clean_text'].str.len()
type_df['text_length'] = type_df['clean_text'].str.len()
length_summary = detection_df.groupby('fraud_label')['text_length'].agg(['count','mean','median','std','min','max']).reset_index()
display(length_summary)
length_summary.to_csv(EDA_ROOT/'텍스트길이_요약.csv', index=False, encoding='utf-8-sig')
fig, axes = plt.subplots(1,3,figsize=(18,5))
sns.countplot(data=detection_df,x='fraud_label',ax=axes[0]); axes[0].set_title('정상상담과 보이스피싱 분포')
sns.boxplot(data=detection_df,x='fraud_label',y='text_length',showfliers=False,ax=axes[1]); axes[1].set_title('정제 텍스트 길이')
sns.countplot(data=type_df,x='supervised_target',ax=axes[2]); axes[2].set_title('보이스피싱 유형 분포')
for ax in axes: ax.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.savefig(EDA_ROOT/'기본_EDA.png',dpi=160,bbox_inches='tight'); plt.show()
direction_summary = amount_df['amount_direction'].fillna('UNKNOWN').value_counts().rename_axis('금액방향').reset_index(name='건수')
purpose_summary = amount_df['amount_purpose'].fillna('UNKNOWN').value_counts().rename_axis('금액용도').reset_index(name='건수')
display(direction_summary); display(purpose_summary.head(15))
direction_summary.to_csv(EDA_ROOT/'금액방향_분포.csv',index=False,encoding='utf-8-sig')
purpose_summary.to_csv(EDA_ROOT/'금액용도_분포.csv',index=False,encoding='utf-8-sig')
fig,axes=plt.subplots(1,2,figsize=(16,5))
sns.countplot(data=amount_df,x='amount_direction',order=amount_df['amount_direction'].value_counts().index,ax=axes[0])
axes[0].set_title('금액 방향 분포'); axes[0].tick_params(axis='x',rotation=25)
top_purpose=amount_df['amount_purpose'].value_counts().head(10).index
sns.countplot(data=amount_df[amount_df['amount_purpose'].isin(top_purpose)],y='amount_purpose',order=top_purpose,ax=axes[1])
axes[1].set_title('금액 용도 상위 10개')
plt.tight_layout(); plt.savefig(EDA_ROOT/'금액방향_용도.png',dpi=160,bbox_inches='tight'); plt.show()
upper=amount_df['amount_10k_krw'].quantile(.99)
plot_amount=amount_df[amount_df['amount_10k_krw'].between(0,upper)]
plt.figure(figsize=(11,5)); sns.histplot(data=plot_amount,x='amount_10k_krw',hue='amount_direction',bins=30)
plt.xlabel('금액(만원)'); plt.title('금액 방향별 분포: 상위 1% 이상치 제외')
plt.tight_layout(); plt.savefig(EDA_ROOT/'금액방향별_금액분포_만원.png',dpi=160); plt.show()

## 6. 학습·검증·최종 테스트 분리

같은 원본 통화는 반드시 하나의 세트에만 들어갑니다. 정상상담의 공식 Validation은 최종 테스트로 보존하고, 보이스피싱은 `group_id` 단위로 대응 분리합니다.

In [ ]:
# 8. 그룹 누출 없는 분리 함수
def random_group_parts(groups, test_ratio=TEST_RATIO, dev_ratio=DEV_RATIO):
    groups=np.array(sorted(pd.Series(groups).dropna().astype(str).unique()))
    train_dev,test=train_test_split(groups,test_size=test_ratio,random_state=SEED)
    train,dev=train_test_split(train_dev,test_size=dev_ratio,random_state=SEED)
    result={g:'TRAIN' for g in train}; result.update({g:'DEV' for g in dev}); result.update({g:'TEST' for g in test})
    return result
def stratified_group_parts(df,label_col):
    grouped=df.groupby('group_id')[label_col].agg(lambda s:s.mode().iloc[0]).reset_index()
    assert df.groupby('group_id')[label_col].nunique().max()==1
    train_dev,test=train_test_split(grouped,test_size=TEST_RATIO,random_state=SEED,stratify=grouped[label_col])
    train,dev=train_test_split(train_dev,test_size=DEV_RATIO,random_state=SEED,stratify=train_dev[label_col])
    result={g:'TRAIN' for g in train.group_id}; result.update({g:'DEV' for g in dev.group_id}); result.update({g:'TEST' for g in test.group_id})
    return result
def check_split(df,label_col):
    assert df.groupby('group_id')['ml_split'].nunique().max()==1, '같은 원본 통화가 여러 세트에 섞였습니다.'
    assert {'TRAIN','DEV','TEST'}.issubset(set(df.ml_split))
    for name in ['TRAIN','DEV','TEST']:
        assert df.loc[df.ml_split.eq(name),label_col].nunique()>=2, f'{name}에 클래스가 하나뿐입니다.'
    return df.groupby(['ml_split',label_col]).size().reset_index(name='건수')

In [ ]:
# 9. 정상 vs 사기 분리
det=detection_df.copy(); det['group_id']=det['group_id'].astype(str); det['ml_split']=''
normal_mask=det.fraud_label.eq('LEGITIMATE_FINANCIAL_CALL')
split_text=det.get('original_split',pd.Series('',index=det.index)).fillna('').astype(str).str.upper()
normal_test=normal_mask & split_text.str.contains('VALID|TEST')
normal_train_pool=normal_mask & ~normal_test
# 공식 Validation 표기가 없을 때만 정상상담도 원본 그룹 단위로 대응 분리합니다.
if normal_test.sum()==0:
    fallback_map=random_group_parts(det.loc[normal_mask,'group_id'].unique())
    det.loc[normal_mask,'ml_split']=det.loc[normal_mask,'group_id'].map(fallback_map)
    normal_train_pool=pd.Series(False,index=det.index)
if normal_train_pool.any():
    normal_train_groups=det.loc[normal_train_pool,'group_id'].unique()
    normal_train,normal_dev=train_test_split(normal_train_groups,test_size=DEV_RATIO,random_state=SEED)
    det.loc[normal_train_pool & det.group_id.isin(normal_train),'ml_split']='TRAIN'
    det.loc[normal_train_pool & det.group_id.isin(normal_dev),'ml_split']='DEV'
    det.loc[normal_test,'ml_split']='TEST'
fraud_groups=det.loc[~normal_mask,'group_id'].unique()
fraud_map=random_group_parts(fraud_groups)
det.loc[~normal_mask,'ml_split']=det.loc[~normal_mask,'group_id'].map(fraud_map)
assert not det.ml_split.eq('').any(), '분리되지 않은 데이터가 있습니다.'
split_summary=check_split(det,'fraud_label'); display(split_summary)
split_summary.to_csv(SPLIT_ROOT/'정상사기_분리요약.csv',index=False,encoding='utf-8-sig')
det[['conversation_id','group_id','fraud_label','ml_split']].to_csv(SPLIT_ROOT/'정상사기_split_id.csv',index=False,encoding='utf-8-sig')

## 7. 길이·비율 편향 민감도 실험

In [ ]:
# 10. 학습용 1:3·1:1 표본과 길이 보정 표본 만들기
FRAUD='VOICE_PHISHING'; NORMAL='LEGITIMATE_FINANCIAL_CALL'
def sample_ratio(df, normal_per_fraud):
    fraud=df[df.fraud_label.eq(FRAUD)]
    normal=df[df.fraud_label.eq(NORMAL)]
    n=min(len(normal),len(fraud)*normal_per_fraud)
    normal=normal.sample(n=n,random_state=SEED) if n else normal
    return pd.concat([fraud,normal]).sample(frac=1,random_state=SEED).reset_index(drop=True)
def center_window(text,target_chars):
    text=str(text)
    if len(text)<=target_chars: return text
    start=(len(text)-target_chars)//2
    return text[start:start+target_chars]
train_all=det[det.ml_split.eq('TRAIN')].copy()
dev_original=det[det.ml_split.eq('DEV')].copy()
target_chars=int(train_all.loc[train_all.fraud_label.eq(FRAUD),'text_length'].median())
target_chars=int(np.clip(target_chars,250,1500))
train_1to3=sample_ratio(train_all,3)
train_1to1=sample_ratio(train_all,1)
train_length_1to1=train_1to1.copy(); dev_length=dev_original.copy()
train_length_1to1['clean_text']=train_length_1to1.clean_text.map(lambda x:center_window(x,target_chars))
dev_length['clean_text']=dev_length.clean_text.map(lambda x:center_window(x,target_chars))
experiment_summary=pd.DataFrame([
 ['원본_1대3',len(train_1to3),train_1to3.fraud_label.value_counts().to_dict(),False],
 ['원본_1대1',len(train_1to1),train_1to1.fraud_label.value_counts().to_dict(),False],
 ['길이보정_1대1',len(train_length_1to1),train_length_1to1.fraud_label.value_counts().to_dict(),True],
],columns=['실험','학습행수','클래스분포','길이보정'])
display(experiment_summary); print('길이 보정 기준:',target_chars,'자')
experiment_summary.to_csv(EDA_ROOT/'비율_길이보정_실험설계.csv',index=False,encoding='utf-8-sig')

## 8. 모델 비교 공통 함수

In [ ]:
# 11. 후보 모델과 평가 함수
def binary_candidates():
    return {
      'Dummy':Pipeline([('tfidf',TfidfVectorizer(max_features=1000)),('model',DummyClassifier(strategy='prior'))]),
      'Word_TFIDF_Logistic':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED))]),
      'Char_TFIDF_Logistic':Pipeline([('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=SEED))]),
      'Char_TFIDF_LinearSVM':Pipeline([('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',LinearSVC(class_weight='balanced',random_state=SEED))]),
      'Char_TFIDF_SGD':Pipeline([('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',SGDClassifier(loss='log_loss',class_weight='balanced',random_state=SEED))]),
      'Word_TFIDF_ComplementNB':Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),('model',ComplementNB())])}
def positive_score(model,x,positive):
    classes=list(model.classes_); idx=classes.index(positive)
    if hasattr(model,'predict_proba'): return model.predict_proba(x)[:,idx]
    score=model.decision_function(x)
    return score if np.ndim(score)==1 and idx==1 else (-score if np.ndim(score)==1 else score[:,idx])
def binary_scores(y_true,y_pred,score,positive=FRAUD):
    p,r,f1,_=precision_recall_fscore_support(y_true,y_pred,average='binary',pos_label=positive,zero_division=0)
    y_bin=(np.asarray(y_true)==positive).astype(int)
    return {'accuracy':accuracy_score(y_true,y_pred),'precision':p,'recall':r,'f1':f1,'pr_auc':average_precision_score(y_bin,score)}
def compare_binary(train,dev,label='fraud_label'):
    rows=[]
    for name,model in binary_candidates().items():
        model.fit(train.clean_text,train[label]); pred=model.predict(dev.clean_text); score=positive_score(model,dev.clean_text,FRAUD)
        rows.append({'모델':name,**binary_scores(dev[label],pred,score)})
        print(name,'완료')
    return pd.DataFrame(rows).sort_values(['pr_auc','f1'],ascending=False).reset_index(drop=True)
def compare_multiclass(train,dev,label):
    rows=[]
    for name,model in binary_candidates().items():
        model.fit(train.clean_text,train[label]); pred=model.predict(dev.clean_text)
        p,r,f1,_=precision_recall_fscore_support(dev[label],pred,average='macro',zero_division=0)
        rows.append({'모델':name,'accuracy':accuracy_score(dev[label],pred),'macro_precision':p,'macro_recall':r,'macro_f1':f1})
    return pd.DataFrame(rows).sort_values('macro_f1',ascending=False).reset_index(drop=True)

## 9. 정상상담 vs 보이스피싱

In [ ]:
# 12. 검증 세트에서 알고리즘과 길이·비율 민감도 비교
base_compare=compare_binary(train_1to3,dev_original)
display(base_compare)
best_detection_name=base_compare.iloc[0]['모델']
sensitivity_rows=[]
for exp_name,train_data,dev_data in [
 ('원본_1대3',train_1to3,dev_original),('원본_1대1',train_1to1,dev_original),('길이보정_1대1',train_length_1to1,dev_length)]:
    model=clone(binary_candidates()[best_detection_name]); model.fit(train_data.clean_text,train_data.fraud_label)
    pred=model.predict(dev_data.clean_text); score=positive_score(model,dev_data.clean_text,FRAUD)
    sensitivity_rows.append({'실험':exp_name,'모델':best_detection_name,**binary_scores(dev_data.fraud_label,pred,score)})
sensitivity_df=pd.DataFrame(sensitivity_rows).sort_values('pr_auc',ascending=False)
display(sensitivity_df)
base_compare.to_csv(REPORT_ROOT/'정상사기_모델비교_검증.csv',index=False,encoding='utf-8-sig')
sensitivity_df.to_csv(REPORT_ROOT/'정상사기_비율_길이_민감도.csv',index=False,encoding='utf-8-sig')
print('주의: 민감도 실험은 편향 확인용이며 최종 테스트 결과로 모델을 다시 고르지 않습니다.')

In [ ]:
# 13. 선택된 모델을 TRAIN+DEV로 재학습한 뒤 최종 TEST를 한 번만 평가
final_train=sample_ratio(det[det.ml_split.isin(['TRAIN','DEV'])],3)
final_test=det[det.ml_split.eq('TEST')].copy()
best_detection=clone(binary_candidates()[best_detection_name])
best_detection.fit(final_train.clean_text,final_train.fraud_label)
det_pred=best_detection.predict(final_test.clean_text); det_score=positive_score(best_detection,final_test.clean_text,FRAUD)
detection_test=binary_scores(final_test.fraud_label,det_pred,det_score)
display(pd.DataFrame([detection_test]))
print(classification_report(final_test.fraud_label,det_pred,zero_division=0))
det_result=final_test[['conversation_id','group_id','fraud_label','clean_text']].copy()
det_result['예측']=det_pred; det_result['보이스피싱점수']=det_score; det_result['정답여부']=det_result.fraud_label.eq(det_pred)
det_result.to_csv(PRED_ROOT/'정상사기_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_detection,MODEL_ROOT/'fraud_detection_best_model.joblib')
labels=[NORMAL,FRAUD]; cm=confusion_matrix(final_test.fraud_label,det_pred,labels=labels)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['정상','보이스피싱'],yticklabels=['정상','보이스피싱'])
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('정상 vs 보이스피싱 최종 테스트')
plt.tight_layout(); plt.savefig(REPORT_ROOT/'정상사기_혼동행렬.png',dpi=160); plt.show()

## 10. 보이스피싱 유형 분류

In [ ]:
# 14. 대출사기형 vs 수사기관사칭형
fraud_type=type_df[type_df.supervised_target.isin(['LOAN_FRAUD','INSTITUTION_IMPERSONATION'])].copy()
# fraud_type_ml에는 group_id가 없으므로 원본 통화 ID인 file_id를 분리 그룹으로 사용합니다.
assert 'file_id' in fraud_type.columns, '원본 통화 분리에 필요한 file_id가 없습니다.'
assert fraud_type['file_id'].notna().all(), 'file_id 결측값이 있습니다.'
fraud_type['group_id']=fraud_type['file_id'].astype(str)
fraud_type['ml_split']=fraud_type.group_id.map(stratified_group_parts(fraud_type,'supervised_target'))
display(check_split(fraud_type,'supervised_target'))
tr=fraud_type[fraud_type.ml_split.eq('TRAIN')]; dv=fraud_type[fraud_type.ml_split.eq('DEV')]; te=fraud_type[fraud_type.ml_split.eq('TEST')]
type_compare=compare_multiclass(tr,dv,'supervised_target'); display(type_compare)
best_type_name=type_compare.iloc[0]['모델']; best_type=clone(binary_candidates()[best_type_name])
best_type.fit(fraud_type[fraud_type.ml_split.isin(['TRAIN','DEV'])].clean_text,fraud_type[fraud_type.ml_split.isin(['TRAIN','DEV'])].supervised_target)
type_pred=best_type.predict(te.clean_text)
p,r,type_f1,_=precision_recall_fscore_support(te.supervised_target,type_pred,average='macro',zero_division=0)
type_test={'accuracy':accuracy_score(te.supervised_target,type_pred),'macro_precision':p,'macro_recall':r,'macro_f1':type_f1}
display(pd.DataFrame([type_test])); print(classification_report(te.supervised_target,type_pred,zero_division=0))
type_result=te[['case_id','group_id','supervised_target','clean_text']].copy(); type_result['예측']=type_pred; type_result['정답여부']=type_result.supervised_target.eq(type_pred)
type_result.to_csv(PRED_ROOT/'사기유형_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
type_compare.to_csv(REPORT_ROOT/'사기유형_모델비교.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_type,MODEL_ROOT/'fraud_type_best_model.joblib')

## 11. 전체·부분 구간 탐지

In [ ]:
# 15. 전체 대화와 임의 구간 탐지
split_lookup=det[['group_id','ml_split']].drop_duplicates(); segment=segment_df.copy(); segment['group_id']=segment.group_id.astype(str)
segment=segment.merge(split_lookup,on='group_id',how='inner'); display(check_split(segment,'fraud_label'))
tr=segment[segment.ml_split.eq('TRAIN')]; dv=segment[segment.ml_split.eq('DEV')]; te_seg=segment[segment.ml_split.eq('TEST')]
segment_compare=compare_binary(sample_ratio(tr,3),dv); display(segment_compare)
best_segment_name=segment_compare.iloc[0]['모델']; best_segment=clone(binary_candidates()[best_segment_name])
segment_train_dev=sample_ratio(segment[segment.ml_split.isin(['TRAIN','DEV'])],3)
best_segment.fit(segment_train_dev.clean_text,segment_train_dev.fraud_label)
seg_pred=best_segment.predict(te_seg.clean_text); seg_score=positive_score(best_segment,te_seg.clean_text,FRAUD)
segment_result=te_seg.copy(); segment_result['예측']=seg_pred; segment_result['보이스피싱점수']=seg_score; segment_result['정답여부']=segment_result.fraud_label.eq(seg_pred)
rows=[]
for (scope,pos),g in segment_result.groupby(['sample_scope','window_position']):
    if g.fraud_label.nunique()<2: continue
    rows.append({'표본범위':scope,'구간위치':pos,'건수':len(g),**binary_scores(g.fraud_label,g['예측'],g['보이스피싱점수'])})
position_df=pd.DataFrame(rows); display(position_df)
segment_result.to_csv(PRED_ROOT/'전체부분구간_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
position_df.to_csv(REPORT_ROOT/'전체부분구간_위치별성능.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_segment,MODEL_ROOT/'segment_detection_best_model.joblib')

## 12. 유사 사건 군집분석

In [ ]:
# 16. K-means·MiniBatch K-means·계층 군집 비교
vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.95,max_features=MAX_FEATURES,sublinear_tf=True)
matrix=vectorizer.fit_transform(cluster_df.clean_text)
n_components=max(2,min(50,matrix.shape[0]-1,matrix.shape[1]-1))
svd=TruncatedSVD(n_components=n_components,random_state=SEED); features=svd.fit_transform(matrix)
rows=[]; outputs={}
for k in range(2,min(8,len(cluster_df)-1)+1):
    candidates={'kmeans':KMeans(n_clusters=k,n_init=20,random_state=SEED),
                'minibatch_kmeans':MiniBatchKMeans(n_clusters=k,n_init=20,batch_size=256,random_state=SEED),
                'agglomerative':AgglomerativeClustering(n_clusters=k)}
    for name,model in candidates.items():
        labels=model.fit_predict(features); sil=silhouette_score(features,labels); stability=np.nan
        if name!='agglomerative':
            other=clone(model).set_params(random_state=SEED+1); stability=adjusted_rand_score(labels,other.fit_predict(features))
        rows.append({'알고리즘':name,'군집수':k,'silhouette':sil,'seed_stability_ari':stability}); outputs[(name,k)]=(model,labels)
cluster_compare=pd.DataFrame(rows).sort_values(['silhouette','seed_stability_ari'],ascending=False); display(cluster_compare.head(12))
best_cluster_row=cluster_compare.iloc[0]; best_cluster_key=(best_cluster_row['알고리즘'],int(best_cluster_row['군집수']))
best_cluster_model,cluster_labels=outputs[best_cluster_key]
cluster_result=cluster_df.copy(); cluster_result['군집ID']=cluster_labels
cluster_result.to_csv(PRED_ROOT/'유사사건_군집결과.csv',index=False,encoding='utf-8-sig')
cluster_compare.to_csv(REPORT_ROOT/'군집모델_비교.csv',index=False,encoding='utf-8-sig')
joblib.dump({'vectorizer':vectorizer,'svd':svd,'model':best_cluster_model,'algorithm':best_cluster_key[0],'cluster_count':best_cluster_key[1]},MODEL_ROOT/'case_clustering_best_model.joblib')
print('군집은 정답 분류가 아니므로 정확도라고 부르지 않습니다.')

## 13. 최종 보고서와 자동 검증

In [ ]:
# 17. 결과 요약·한계·실행기록 저장
best_models=pd.DataFrame([
 ['정상상담 vs 보이스피싱',best_detection_name,'PR-AUC',detection_test['pr_auc']],
 ['보이스피싱 유형',best_type_name,'Macro F1',type_test['macro_f1']],
 ['전체·부분 구간',best_segment_name,'평균 PR-AUC',position_df.pr_auc.mean() if len(position_df) else np.nan],
 ['유사 사건 군집',f'{best_cluster_key[0]} (k={best_cluster_key[1]})','Silhouette',best_cluster_row.silhouette]
],columns=['시나리오','선정모델','최종지표','최종점수'])
display(best_models); best_models.to_csv(REPORT_ROOT/'시나리오별_최종모델.csv',index=False,encoding='utf-8-sig')
report=[
 '# 3단계 v5 머신러닝 분석 결과','', '## Summary','',
 f'- 정상·사기 모델: {best_detection_name}',f"- 최종 PR-AUC: {detection_test['pr_auc']:.4f}",f"- 보이스피싱 Recall: {detection_test['recall']:.4f}",
 f'- 사기유형 모델: {best_type_name} / Macro F1 {type_test["macro_f1"]:.4f}',
 f'- 구간탐지 모델: {best_segment_name}',f'- 군집: {best_cluster_key[0]} / k={best_cluster_key[1]}','',
 '## v4 편향 점검','',f'- 길이 보정 기준: {target_chars}자','- 1:3, 1:1, 길이보정 1:1 결과는 검증 세트에서 비교함','- 최종 테스트는 선택된 모델에 한 번만 사용함','',
 '## 해석 시 주의','',
 '- 정상상담과 보이스피싱은 출처와 원본 편집 방식이 달라 현재 점수는 두 공개 코퍼스 구분 성능입니다.',
 '- 길이 보정 후 성능이 크게 낮아지면 모델이 사기 표현 외에 텍스트 길이를 이용했을 가능성이 있습니다.',
 '- 금액 방향과 용도는 자동 추출 SILVER 라벨이며 실제 피해액을 뜻하지 않습니다.',
 '- 실제 피해 여부 정답이 없으므로 피해 발생 확률로 해석할 수 없습니다.'
]
(REPORT_ROOT/'03_ml_analysis_report_v4.md').write_text('\n'.join(report),encoding='utf-8')
manifest={'version':'v5','seed':SEED,'dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
          'target_chars':target_chars,'best_models':best_models.to_dict(orient='records'),
          'group_leakage_checks_passed':True,'test_used_once_for_final_model':True,
          'saved_models':[p.name for p in MODEL_ROOT.glob('*.joblib')]}
(REPORT_ROOT/'ml_run_manifest_v4.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
assert len(list(MODEL_ROOT.glob('*.joblib')))==4
assert det.groupby('group_id').ml_split.nunique().max()==1
assert fraud_type.groupby('group_id').ml_split.nunique().max()==1
assert segment.groupby('group_id').ml_split.nunique().max()==1
print('3단계 v4 정상 완료:',OUTPUT_ROOT)

---

# PART D — v5 통합 시각화 강화

앞선 v4 기반 분석 결과를 한 화면에서 검토할 수 있도록 핵심 분포와 성능을 통합 시각화합니다. 기존 분석값은 변경하지 않습니다.


In [ ]:
# v5 통합 핵심 시각화
V5_VIS_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v5' / '04_통합시각화'
V5_VIS_ROOT.mkdir(parents=True, exist_ok=True)

# PART C에서 정제된 데이터프레임을 우선 사용합니다.
viz_detection = detection_df.copy()
viz_type = type_df.copy()
viz_amount = amount_df.copy()

fig, axes = plt.subplots(2, 3, figsize=(21, 13))

# 1. 정상·사기 클래스 구성
class_order = viz_detection['fraud_label'].value_counts().index
sns.countplot(data=viz_detection, x='fraud_label', order=class_order, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('정상 금융상담과 보이스피싱 표본 수')
axes[0, 0].set_xlabel('분류'); axes[0, 0].set_ylabel('통화 수')
axes[0, 0].tick_params(axis='x', rotation=15)

# 2. 텍스트 길이 분포
if 'text_length' not in viz_detection.columns:
    viz_detection['text_length'] = viz_detection['clean_text'].fillna('').str.len()
sns.boxplot(data=viz_detection, x='fraud_label', y='text_length', showfliers=False, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('정상·사기 텍스트 길이 분포(이상치 제외)')
axes[0, 1].set_xlabel('분류'); axes[0, 1].set_ylabel('문자 수')
axes[0, 1].tick_params(axis='x', rotation=15)

# 3. 보이스피싱 유형 구성
sns.countplot(data=viz_type, y='supervised_target', order=viz_type['supervised_target'].value_counts().index,
              ax=axes[0, 2], palette='Set1')
axes[0, 2].set_title('보이스피싱 유형 분포')
axes[0, 2].set_xlabel('사건 수'); axes[0, 2].set_ylabel('유형')

# 4. 금액 방향 상위 분포
direction_order = viz_amount['amount_direction'].fillna('MISSING').value_counts().head(8).index
sns.countplot(data=viz_amount.assign(amount_direction=viz_amount['amount_direction'].fillna('MISSING')),
              y='amount_direction', order=direction_order, ax=axes[1, 0], color='#4c78a8')
axes[1, 0].set_title('금액 방향 분포 상위 8개')
axes[1, 0].set_xlabel('이벤트 수'); axes[1, 0].set_ylabel('금액 방향')

# 5. 규칙 기반 표현 밀도 차이
# 주의: 행동 자체가 아니라 정규식으로 검출된 표현의 코퍼스 간 차이입니다.
if 'stats_df' in globals() and len(stats_df):
    diff_col = '평균밀도차이' if '평균밀도차이' in stats_df.columns else '평균밀도차이_사기빼기정상'
    feature_plot = stats_df.sort_values(diff_col).copy()
    colors = np.where(feature_plot[diff_col] >= 0, '#d95f5f', '#4c78a8')
    axes[1, 1].barh(feature_plot['특징'], feature_plot[diff_col], color=colors)
    axes[1, 1].axvline(0, color='black', linewidth=1)
    axes[1, 1].set_title('정규식 표현 밀도 차이\n(보이스피싱 − 정상상담)')
    axes[1, 1].set_xlabel('1,000자당 평균 검출 차이')
    axes[1, 1].set_ylabel('표현 규칙')
else:
    axes[1, 1].text(0.5, 0.5, '해석형 특징 결과 없음', ha='center', va='center')
    axes[1, 1].set_axis_off()

# 6. 정상·사기 후보 모델 검증 성능
if 'base_compare' in globals() and len(base_compare):
    metric_cols = [c for c in ['accuracy', 'precision', 'recall', 'f1', 'pr_auc'] if c in base_compare.columns]
    model_heatmap = base_compare.set_index('모델')[metric_cols]
    sns.heatmap(model_heatmap, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1, ax=axes[1, 2])
    axes[1, 2].set_title('정상·사기 후보 모델 검증 성능')
    axes[1, 2].set_xlabel('평가지표'); axes[1, 2].set_ylabel('모델')
else:
    axes[1, 2].text(0.5, 0.5, '모델 비교 결과 없음', ha='center', va='center')
    axes[1, 2].set_axis_off()

fig.suptitle('보이스피싱 통합 분석 v5 — 핵심 데이터·모델 진단', fontsize=18, y=1.01)
plt.tight_layout()
plt.savefig(V5_VIS_ROOT / '01_v5_핵심분석_대시보드.png', dpi=180, bbox_inches='tight')
plt.show()

# 클래스별 텍스트 길이 ECDF: 박스플롯으로 보이지 않는 전체 분포를 보완
plt.figure(figsize=(11, 6))
sns.ecdfplot(data=viz_detection, x='text_length', hue='fraud_label')
plt.xscale('log')
plt.title('정상·사기 텍스트 길이 누적분포(로그 축)')
plt.xlabel('텍스트 길이(문자 수, 로그 축)'); plt.ylabel('누적비율')
plt.tight_layout()
plt.savefig(V5_VIS_ROOT / '02_정상사기_텍스트길이_ECDF.png', dpi=180, bbox_inches='tight')
plt.show()

print('v5 통합 시각화 저장 완료:', V5_VIS_ROOT)


---

# 실행 완료 확인

PART A, PART B, PART C가 오류 없이 끝났는지 각 파트 마지막의 완료 메시지와 Google Drive의 `머신러닝_분석결과_v5` 폴더를 확인하세요.
